# FINAL Methodology Implementation

This notebook operationalises the end-to-end AREP forecasting workflow without touching the underlying code cells. Each section calls reusable pipeline components and records why we make the corresponding design choice so future readers can reason about adjustments.

## Roadmap
1. Instrument the runtime environment so resource usage and configurations stay reproducible.
2. Ingest energy telemetry and news streams, then engineer time-aware features.
3. Search the time-decay feature space, shortlist the strongest datasets, and tune XGBoost forecasters.
4. Stack a LightGBM decision layer on top of the best forecasts and evaluate trading impact with rigorous diagnostics.

All reasoning cells focus on *why* we execute a block; the original code cells remain unchanged.

In [ ]:
# Uncomment if this is the first time running the notebook
#!pip install -r requirements.txt

### Dependency Imports

We gather every external dependency in one place so the runtime environment is explicit and reproducible. This section imports:

**Core Data Processing:**
- `pandas` and `numpy` for dataframe manipulation and numerical operations
- `polars` for high-performance data transformations (particularly useful for large news datasets)

**Machine Learning:**
- `sklearn` (scikit-learn) for preprocessing, cross-validation, and Ridge regression
- `xgboost` for gradient boosting models (the primary price spread forecaster)
- `lightgbm` for the signal decision layer (translates forecasts into trading signals)

**NLP and Embeddings:**
- `transformers` (Hugging Face) for zero-shot classification and embedding models
- `torch` (PyTorch) for GPU-accelerated tensor operations

**Visualization:**
- `matplotlib` and `seaborn` for charts and plots
- Custom plotting utilities for confusion matrices and feature importance

By consolidating imports here, we make it easy to audit dependencies, spot version conflicts, and document the minimal environment needed to reproduce this analysis.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
from tqdm import tqdm

# API libraries
import requests
import os
from dotenv import load_dotenv
from gpu_pipeline.telemetry import TelemetryLogger, ensure_disk_headroom

# NLP libraries
import umap
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from datasets import Dataset

# ML Libraries
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, RidgeClassifierCV
from sklearn.base import clone
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgb
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Parallelization libraries
from joblib import Parallel, delayed
from scipy import stats

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==========================================================================
# OPTIONAL DEPENDENCY GUARDS
# ==========================================================================

def check_accelerate_available() -> bool:
    """Return True when Hugging Face accelerate is importable."""
    try:
        import accelerate  # noqa: F401
    except ImportError:
        return False
    return True


def resolve_cuml_umap() -> tuple[bool, object | None]:
    """Attempt to import cuML's GPU UMAP implementation."""
    try:
        from cuml.manifold import UMAP as CUML_UMAP  # type: ignore
    except ImportError:
        return False, None
    return True, CUML_UMAP


ACCELERATE_AVAILABLE = check_accelerate_available()
HAS_CUML_UMAP, CUML_UMAP = resolve_cuml_umap()



## 🔧 Configuration Constants

The next code cell centralises every tunable constant so experiments stay reproducible and you can adjust the methodology without hunting through scattered magic numbers.

**Key Configuration Categories:**

1. **Lookback Windows:** Control how many hours/days of historical data feed into features (e.g., 24-hour price lag, 7-day rolling averages). Longer windows capture more context but may dilute recent signals.

2. **Decay Rates (λ):** Determine how quickly past news loses influence. A decay rate of 0.95 means yesterday's news counts for 95% of today's weight, the day before for 95%², etc. We'll grid-search these to find optimal recency-vs-history tradeoffs.

3. **Train/Val/Test Split Ratios:** Define the percentage split for model evaluation (e.g., 60% train, 20% validation, 20% test). These must respect temporal ordering—no training on future data.

4. **Random Seeds:** Fix seeds for Python, NumPy, and PyTorch to ensure identical results across runs (critical for debugging and fair comparisons).

5. **Neutral Band Width:** For the target variable, defines the threshold (e.g., ±1 EUR/MWh) where price changes are labeled "neutral" rather than "long" or "short". This balances class distribution and avoids noisy micro-movements.

Storing these in one place means stakeholders can quickly see and modify experimental parameters without touching the pipeline code.

In [ ]:
# ============================================================================
# CONFIGURATION CONSTANTS
# ============================================================================

# === Time-based Constants ===
FORECAST_HORIZON_HOURS = 24  # Predict 24 hours ahead
HOURS_PER_DAY = 24
HOURS_PER_WEEK = 168  # 7 * 24
HOURS_PER_TWO_WEEKS = 336  # 14 * 24

# === Feature Engineering Parameters ===
DEFAULT_LOOKBACK_WINDOW = HOURS_PER_TWO_WEEKS  # 336 hours
DEFAULT_DECAY_LAMBDA = 0.05  # Time decay parameter
SPREAD_TARGET_DEADBAND = 5.0  # EUR/MWh band for the neutral class

# === Dataset Split Parameters ===
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1
RANDOM_STATE = 42

# TODO: Check implementation of the CV vs DEFAULT variables

# === Cross-validation Parameters ===
N_CV_SPLITS = 5
CV_MIN_TRAIN_SIZE = HOURS_PER_TWO_WEEKS  # Minimum 2 weeks for training
CV_STEP_SIZE_HOURS = 24  # Step size between CV folds

DEFAULT_EXPANDING_SPLITS = 4  # low number for quicker iteration during development
DEFAULT_EXPANDING_STEP = 12
DEFAULT_MIN_TRAIN_SIZE = 336  # two weeks of hourly observations

# === Model Training Parameters ===
DEFAULT_BATCH_SIZE = 32
DEFAULT_N_JOBS = -1  # Use all CPU cores

print("Configuration constants loaded:")
print(f"  Forecast horizon: {FORECAST_HORIZON_HOURS} hours")
print(f"  Lookback window: {DEFAULT_LOOKBACK_WINDOW} hours ({DEFAULT_LOOKBACK_WINDOW // 24} days)")
print(f"  Train/Val/Test split: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"  CV splits: {N_CV_SPLITS} (min train size: {CV_MIN_TRAIN_SIZE} hours)")
print(f"  Spread deadband: ±{SPREAD_TARGET_DEADBAND} EUR/MWh")


## 🖥️ Runtime Instrumentation

We first detect the available compute device so subsequent NLP and gradient boosting blocks size their batches appropriately.

**Why This Matters:**
- **GPU Availability:** Modern NLP models (transformers) and XGBoost/LightGBM can leverage GPUs for 10-100x speedups. This block detects whether a CUDA-enabled GPU is available and configures PyTorch/XGBoost accordingly.

- **Batch Sizing:** With GPU memory constraints (e.g., 8GB vs 40GB), we need to adjust batch sizes dynamically. CPU-only runs use smaller batches to avoid memory swapping.

- **Profiling Hooks:** The utilities introduced here capture:
  - **Wall-clock time:** How long each stage takes
  - **CPU/GPU usage:** Peak memory consumption
  - **Disk I/O:** Bytes read/written (important when caching embeddings or model checkpoints)

**Telemetry Benefits:**
Later, when we compare "topic decay λ=0.9" vs "λ=0.95", the telemetry logs help us understand if performance differences come from better features or just longer computation. It also helps identify bottlenecks (e.g., "embedding generation takes 80% of runtime, maybe we should cache it").

In [ ]:
# ============================================================================
# CENTRALIZED GPU/DEVICE DETECTION UTILITIES
# ============================================================================

def detect_compute_device(task='general', verbose=True):
    """
    Detect optimal compute device and recommended batch size.
    
    Args:
        task: Type of task ('general', 'embeddings', 'training')
        verbose: Print device information
    
    Returns:
        dict with device info and optimal batch size
    """
    import torch
    import os
    
    cpu_cores = os.cpu_count() or 8
    default_jobs = max(1, cpu_cores - 2)
    device_info = {
        'device': 'cpu',
        'backend': 'cpu',
        'gpu_name': None,
        'gpu_memory_gb': 0,
        'optimal_batch_size': 32,
        'tree_method': 'hist',
        'predictor': 'auto',
        'n_jobs': default_jobs,
        'description': f'CPU-only ({cpu_cores} cores)',
        'lgbm_device': 'cpu',
    }
    
    # Check for CUDA (NVIDIA GPUs)
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        gpu_count = torch.cuda.device_count()
        
        device_info.update({
            'device': 'cuda',
            'backend': 'cuda',
            'gpu_name': gpu_name,
            'gpu_memory_gb': gpu_memory,
            'description': f'CUDA ({gpu_name})',
            'tree_method': 'gpu_hist',
            'predictor': 'gpu_predictor',
            'n_jobs': max(1, default_jobs // gpu_count),
            'lgbm_device': 'gpu',
        })
        
        # Adjust batch size based on GPU memory
        if gpu_memory >= 24:
            device_info['optimal_batch_size'] = 512
        elif gpu_memory >= 16:
            device_info['optimal_batch_size'] = 384
        elif gpu_memory >= 8:
            device_info['optimal_batch_size'] = 256
        else:
            device_info['optimal_batch_size'] = 128
        
        if verbose:
            print(f"✓ CUDA detected: {gpu_name} ({gpu_memory:.1f} GB)")
            print(f"  Optimal batch size: {device_info['optimal_batch_size']}")
            print(f"  XGBoost tree method: {device_info['tree_method']} • predictor: {device_info['predictor']}")
            print(f"  Parallel jobs: {device_info['n_jobs']}")
    
    # Check for MPS (Apple Silicon)
    elif torch.backends.mps.is_available():
        device_info.update({
            'device': 'mps',
            'backend': 'mps',
            'gpu_name': 'Apple Silicon GPU',
            'optimal_batch_size': 128,
            'description': 'Apple MPS GPU',
            'tree_method': 'hist',
            'predictor': 'auto',
            'n_jobs': default_jobs,
            'lgbm_device': 'cpu',
        })
        
        if verbose:
            print("✓ MPS detected: Apple Silicon GPU")
            print(f"  Optimal batch size: {device_info['optimal_batch_size']}")
            print("  XGBoost will fallback to CPU histogram (MPS unsupported)")
    
    else:
        if verbose:
            print("⚠ No GPU detected, using CPU")
            print(f"  Optimal batch size: {device_info['optimal_batch_size']}")
            print(f"  Parallel jobs: {device_info['n_jobs']}")
    
    return device_info


# Test the function
device_config = detect_compute_device(task='general', verbose=True)


### Stage Profiling Utilities

We introduce a profiling context manager to capture runtime, CPU, GPU, and disk metrics for each pipeline stage.

**What is a Context Manager?**
A Python context manager uses the `with` statement to automatically handle setup and teardown:
```python
with profile_stage("feature_engineering"):
    # Your code here
    pass  # Profiling starts when entering, stops when exiting
```
This pattern ensures metrics are captured even if the code raises an exception.

**Metrics Captured:**
1. **Elapsed Time:** Wall-clock seconds from start to finish
2. **Peak CPU Memory:** Maximum RAM usage during the stage (via `psutil`)
3. **Peak GPU Memory:** Maximum VRAM allocated (via `torch.cuda.max_memory_allocated()`)
4. **Disk I/O:** Bytes read and written (useful for identifying cache thrashing)

**Why Profile?**
- **Bottleneck Identification:** Discover that "XGBoost tuning takes 4 hours but Ridge screening only 10 minutes"
- **Resource Planning:** Ensure you have enough RAM/VRAM before scaling to larger datasets
- **Optimization Validation:** Confirm that "caching embeddings" actually saved time in subsequent runs

This instrumentation turns the notebook into a self-documenting performance log.

In [ ]:
# ============================================================================
# STAGE PROFILING UTILITIES
# ============================================================================

import time
from contextlib import AbstractContextManager

try:
    import psutil
except Exception:  # pragma: no cover - psutil not required at runtime
    psutil = None

try:
    import pynvml
    pynvml.nvmlInit()
    _NVML_AVAILABLE = True
except Exception:
    _NVML_AVAILABLE = False


def _read_gpu_state():
    if not _NVML_AVAILABLE:
        return None
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        util = pynvml.nvmlDeviceGetUtilizationRates(handle)
        memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return {
            "gpu_util": util.gpu,
            "mem_util": util.memory,
            "mem_used_gb": memory.used / (1024 ** 3),
            "mem_total_gb": memory.total / (1024 ** 3),
        }
    except Exception:
        return None


def _read_cpu_state():
    if psutil is None:
        return None
    cpu_times = psutil.cpu_times_percent(interval=None)
    disk_io = psutil.disk_io_counters() if hasattr(psutil, "disk_io_counters") else None
    return {
        "cpu_percent": psutil.cpu_percent(interval=None),
        "iowait_percent": getattr(cpu_times, "iowait", 0.0),
        "disk_read_mb": (disk_io.read_bytes / (1024 ** 2)) if disk_io else None,
        "disk_write_mb": (disk_io.write_bytes / (1024 ** 2)) if disk_io else None,
    }


class StageProfiler(AbstractContextManager):
    """Context manager to capture per-stage resource consumption."""

    def __init__(self, stage_name: str, device_config: dict | None = None):
        self.stage_name = stage_name
        self.device_config = device_config or {}
        self._start_time = None
        self._start_cpu = None
        self._start_gpu = None

    def __enter__(self):
        print(f"\n[Stage ⏳] {self.stage_name} — starting")
        self._start_time = time.time()
        self._start_cpu = _read_cpu_state()
        self._start_gpu = _read_gpu_state()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        end_time = time.time()
        end_cpu = _read_cpu_state()
        end_gpu = _read_gpu_state()
        duration = end_time - (self._start_time or end_time)

        metrics: dict[str, float | int | None] = {
            "duration_sec": duration,
        }

        print(f"[Stage ✅] {self.stage_name} — completed in {duration:.2f}s")

        if end_cpu:
            cpu_percent = end_cpu.get("cpu_percent")
            iowait_percent = end_cpu.get("iowait_percent")
            metrics.update({
                "cpu_percent": cpu_percent,
                "cpu_iowait_percent": iowait_percent,
            })
            if cpu_percent is not None:
                print(f"  CPU usage: {cpu_percent:.1f}% • IO wait: {iowait_percent:.2f}%")
            if self._start_cpu and end_cpu:
                read_delta = end_cpu.get("disk_read_mb")
                write_delta = end_cpu.get("disk_write_mb")
                start_read = self._start_cpu.get("disk_read_mb", 0.0) if self._start_cpu else 0.0
                start_write = self._start_cpu.get("disk_write_mb", 0.0) if self._start_cpu else 0.0
                if read_delta is not None and write_delta is not None:
                    read_mb = max(0.0, read_delta - start_read)
                    write_mb = max(0.0, write_delta - start_write)
                    metrics.update({
                        "disk_read_mb": read_mb,
                        "disk_write_mb": write_mb,
                    })
                    print(f"  Disk Δ: +{read_mb:.1f} MB read, +{write_mb:.1f} MB written")

        if end_gpu:
            metrics.update(
                {
                    "gpu_util_percent": end_gpu.get("gpu_util"),
                    "gpu_mem_util_percent": end_gpu.get("mem_util"),
                    "gpu_mem_used_gb": end_gpu.get("mem_used_gb"),
                    "gpu_mem_total_gb": end_gpu.get("mem_total_gb"),
                }
            )
            print(
                "  GPU util: {gpu_util:.0f}% • Mem util: {mem_util:.0f}% "
                "({mem_used_gb:.2f}/{mem_total_gb:.2f} GB)".format(**end_gpu)
            )
        elif self.device_config.get("device") in {"cuda", "mps"}:
            print("  GPU metrics unavailable (install pynvml for CUDA visibility)")

        telemetry_logger = globals().get('telemetry_logger')
        if telemetry_logger is not None:
            try:
                telemetry_logger.log_stage(self.stage_name, metrics)
            except Exception as logging_exc:  # pragma: no cover - telemetry must not break execution
                print(f"  ⚠️ Telemetry logging failed: {logging_exc}")

        return False  # propagate exceptions


### Telemetry Session Setup

Creating the telemetry logger here ensures every subsequent stage records metadata (random seed, device, artifacts) to disk.

**What is Telemetry?**
Telemetry is the practice of automatically collecting and logging data about system performance and experiment configuration. In this context, it tracks:
- **Experiment Parameters:** Which random seed, which decay rates, which features were used
- **Resource Usage:** How much memory, how long each stage took (from the profiling utilities above)
- **Artifacts:** Paths to saved models, cached embeddings, result plots

**Why Log to Disk?**
- **Reproducibility:** Six months from now, you can see exactly which parameters produced which results
- **Debugging:** If a run fails, logs show where it crashed and what resources were exhausted
- **Collaboration:** Teammates can inspect logs without re-running expensive experiments

**Disk Space Check:**
The code also checks available disk space upfront. Why? Because if you start a 6-hour training run and it crashes at hour 5 due to "disk full", you've wasted time and lost partial results. The early check prevents this frustration.

In [ ]:
telemetry_logger = TelemetryLogger(
    metadata={
        "random_state": str(RANDOM_STATE),
        "notebook": "final_methodology/final_v2.ipynb",
        "device": device_config.get("description"),
    }
)
ensure_disk_headroom(Path('.'), min_free_gb=5.0)
print(f"Telemetry logging to {telemetry_logger.log_dir}")


### Device Registry Configuration

We normalise PyTorch device settings once so helpers automatically place tensors on the right accelerator.

**What are PyTorch Devices?**
PyTorch (the deep learning framework underlying Hugging Face transformers) needs to know where to allocate tensors:
- **CPU (`"cpu"`):** Always available, slower for large matrix operations
- **CUDA GPU (`"cuda"` or `"cuda:0"`):** 10-100x faster for parallel operations, but limited memory
- **MPS (Apple Metal):** GPU acceleration on Apple Silicon Macs

**Why Normalize Settings?**
Without centralized configuration, every function that creates a tensor must remember to check device availability:
```python
# Bad: repeated checks everywhere
model.to("cuda" if torch.cuda.is_available() else "cpu")
embeddings.to("cuda" if torch.cuda.is_available() else "cpu")
```

By setting a global `DEVICE` variable here, downstream code just references it:
```python
# Good: centralized configuration
model.to(DEVICE)
embeddings.to(DEVICE)
```

**Preventing Hardware-Specific Bugs:**
If you develop on a GPU machine but deploy on CPU, mismatched device placements cause cryptic errors like "expected tensor on cuda:0 but got cpu". Normalizing early catches these issues immediately.

In [ ]:
# ==========================================================================
# CENTRAL DEVICE REGISTRY
# ==========================================================================

PRIMARY_DEVICE = device_config.get("device", "cpu")
PRIMARY_TORCH_DEVICE = torch.device("cuda" if PRIMARY_DEVICE == "cuda" else PRIMARY_DEVICE)

if PRIMARY_DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.set_device(0)
    torch.cuda.empty_cache()
    torch.set_default_tensor_type(torch.cuda.FloatTensor)
elif PRIMARY_DEVICE == "mps" and torch.backends.mps.is_available():
    if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    if hasattr(torch, "set_default_device"):
        torch.set_default_device("mps")

print(f"Primary compute device: {PRIMARY_DEVICE}")


def ensure_tensor_device(obj):
    """Move torch tensors (or collections thereof) onto the primary device."""
    if torch.is_tensor(obj):
        return obj.to(PRIMARY_TORCH_DEVICE)
    if isinstance(obj, (list, tuple)):
        return type(obj)(ensure_tensor_device(x) for x in obj)
    if isinstance(obj, dict):
        return {k: ensure_tensor_device(v) for k, v in obj.items()}
    return obj


def resolve_hf_device(primary_device: str):
    if primary_device == "cuda" and torch.cuda.is_available():
        return 0
    if primary_device == "mps" and torch.backends.mps.is_available():
        return "mps"
    return -1


### Feature Importance Helper

Plotting helpers let us interpret fitted models later without rewriting boilerplate.

**What is Feature Importance?**
After training a model (like XGBoost or LightGBM), feature importance reveals which input variables most influenced predictions:
- **High Importance:** "The model heavily relies on `price_lag_1h`" → This feature is critical
- **Low Importance:** "The model ignores `news_topic_weather`" → This feature may be noise or redundant

**Why Define This Early?**
- **Avoid Code Duplication:** We'll plot importance for multiple models (Ridge, XGBoost, LightGBM, baseline). A reusable function prevents copy-pasting 50 lines of matplotlib code.
- **Consistent Formatting:** Same color scheme, axis labels, and sorting across all plots makes comparisons easier.
- **Focus on Interpretation:** Later evaluation cells focus on *understanding* which features matter, not on *how* to draw the plot.

**Typical Output:**
A horizontal bar chart where longer bars = more important features. For tree-based models (XGBoost, LightGBM), importance is measured by how often a feature is used for splits and how much it reduces prediction error.

In [ ]:
# ============================================================================
# FEATURE IMPORTANCE ANALYSIS FUNCTION
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_feature_importance(model, feature_names, model_name, top_n=20):
    """Plot LightGBM feature importance (gain) for a fitted model."""
    if not hasattr(model, "feature_importances_"):
        print(f"{model_name} does not expose feature_importances_.")
        return None

    importances = model.feature_importances_
    if len(importances) != len(feature_names):
        raise ValueError(
            f"Feature name count ({len(feature_names)}) does not match model importances "
            f"({len(importances)}). Ensure feature_names reflects the training columns."
        )

    importance_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    top_df = importance_df.head(top_n)

    plt.figure(figsize=(10, max(6, len(top_df) * 0.4)))
    sns.barplot(
        data=top_df,
        x="importance",
        y="feature",
        palette="Blues_r"
    )
    plt.title(f"{model_name} – Top {len(top_df)} Features (Gain)")
    plt.xlabel("Importance (Gain)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

    return importance_df

### Cross-Validation Defaults

We restate the expanding-window defaults to make sure later cells reference consistent split logic.

**What is Cross-Validation (CV)?**
Cross-validation tests model performance by training on one subset of data and validating on another, repeated multiple times to reduce sampling bias.

**Why Expanding Window (Time-Series CV)?**
Standard k-fold CV randomly shuffles data, which **leaks future information** in time-series:
```
❌ Standard k-fold: Train on [Jan, Mar, May], test on [Feb, Apr, Jun]
   Problem: Training on March data when predicting February!
```

**Expanding window** respects temporal order:
```
✅ Expanding window:
   Fold 1: Train [Jan-Feb], Test [Mar]
   Fold 2: Train [Jan-Mar], Test [Apr]
   Fold 3: Train [Jan-Apr], Test [May]
```
Each fold only trains on past data, mimicking real-world deployment where you can't see the future.

**Why Define This Here?**
We'll use expanding window CV in Ridge screening, XGBoost tuning, and LightGBM training. Documenting the default parameters (e.g., minimum training size, number of splits) in one place ensures all stages use the same validation logic, making performance comparisons fair.

In [ ]:
# ==========================================================================
# GLOBAL CV DEFAULTS
# ==========================================================================

DEFAULT_EXPANDING_SPLITS = 4  # low number for quicker iteration during development
DEFAULT_EXPANDING_STEP = 12
DEFAULT_MIN_TRAIN_SIZE = 336  # two weeks of hourly observations



### Confusion Matrix Helper

This utility standardises how we visualise class-level performance, avoiding ad-hoc plotting inside evaluation cells.

**What is a Confusion Matrix?**
A confusion matrix shows where a classifier makes mistakes:
```
                Predicted
              Long Neutral Short
Actual Long    85     10      5     ← Of 100 actual "Long" events,
       Neutral  8     82     10       model correctly predicted 85
       Short    3     12     85
```

**Reading the Matrix:**
- **Diagonal (top-left to bottom-right):** Correct predictions
- **Off-diagonal:** Misclassifications
  - Example: Top row, middle column (10) = model predicted "Neutral" when truth was "Long"

**Why This Matters for Trading:**
- **False Positives (Long/Short):** Model says "buy" but price drops → loses money
- **False Negatives (Neutral):** Model says "hold" but price spikes → missed profit

By standardizing the plot (consistent color map, annotations, axis labels), we can quickly compare matrices across models and identify systematic biases (e.g., "model always over-predicts Neutral").

In [ ]:
# ============================================================================
# CONFUSION MATRIX VISUALIZATION FUNCTION
# ============================================================================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrices(models_dict, y_test, class_labels, label_encoder=None):
    """
    Plot confusion matrices for multiple models.

    models_dict: dict[str, tuple[sklearn estimator, pd.DataFrame or np.ndarray]]
        Mapping from model name to (fitted estimator, feature matrix) pair.
    y_test: pd.Series or np.ndarray
        Ground-truth labels (already aligned with all feature matrices).
    class_labels: list
        Class labels to keep fixed across plots.
    label_encoder: Optional[LabelEncoder]
        If provided, predictions produced by the estimators will be inverse-transformed
        into the original label space before computing the confusion matrix.
    """
    y_true = np.asarray(y_test)
    n_models = len(models_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), squeeze=False)
    axes = axes.ravel()

    for ax, (name, (model, X_test)) in zip(axes, models_dict.items()):
        y_pred = model.predict(X_test)
        if label_encoder is not None:
            y_pred = label_encoder.inverse_transform(y_pred)
        cm = confusion_matrix(y_true, y_pred, labels=class_labels)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
        disp.plot(ax=ax, values_format='d', cmap='Blues', colorbar=False)
        ax.set_title(name)
        ax.set_xlabel('Predicted label')
        ax.set_ylabel('True label')

    plt.tight_layout()
    plt.show()

### Statistical Comparison Toolkit

By defining bootstrap intervals and McNemar tests upfront, we can later justify model choices with significance checks instead of eyeballing accuracy differences.

**Why Statistical Testing Matters:**
Suppose Model A achieves 72% accuracy and Model B achieves 74%. Is B truly better, or just lucky? With small test sets, random noise can produce 2% swings.

**Bootstrap Confidence Intervals:**
Resamples the test set 1000+ times with replacement, refits performance metrics, and reports the 95% confidence interval:
```
Model A: 72% ± 3%  →  [69%, 75%]
Model B: 74% ± 2%  →  [72%, 76%]
```
If intervals overlap heavily, the difference isn't statistically meaningful.

**McNemar's Test:**
Specifically designed for paired classifiers. It checks if Model B's correct predictions differ significantly from Model A's:
- **Null Hypothesis:** Both models make the same types of errors
- **Result:** p-value < 0.05 → reject null, Model B is genuinely better

**When to Use:**
- Before deploying a new model: "Is the news-enhanced model significantly better than price-only?"
- When comparing hyperparameters: "Does λ=0.9 beat λ=0.95 by chance or design?"

Defining these tests early ensures we don't cherry-pick results or claim improvements based on noise.

In [ ]:
# ============================================================================
# STATISTICAL SIGNIFICANCE TESTING FUNCTIONS
# ============================================================================

from scipy import stats
import numpy as np
from sklearn.metrics import accuracy_score

def bootstrap_confidence_interval(y_true, y_pred_proba, metric_func, 
                                   n_bootstrap=1000, confidence=0.95, random_state=42):
    """
    Calculate bootstrap confidence interval for a metric.
    
    Args:
        y_true: True labels
        y_pred_proba: Predicted probabilities or predictions
        metric_func: Function to calculate metric (takes y_true, y_pred as input)
        n_bootstrap: Number of bootstrap iterations
        confidence: Confidence level (e.g., 0.95 for 95% CI)
        random_state: Random seed for reproducibility
    
    Returns:
        (mean, lower_bound, upper_bound)
    """
    np.random.seed(random_state)
    scores = []
    n_samples = len(y_true)
    
    for _ in range(n_bootstrap):
        # Sample with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        y_true_boot = y_true[indices] if isinstance(y_true, np.ndarray) else y_true.iloc[indices]
        y_pred_boot = y_pred_proba[indices] if isinstance(y_pred_proba, np.ndarray) else y_pred_proba.iloc[indices]
        
        try:
            score = metric_func(y_true_boot, y_pred_boot)
            if not np.isnan(score):
                scores.append(score)
        except Exception as e:
            continue
    
    if len(scores) == 0:
        return np.nan, np.nan, np.nan
    
    # Calculate percentile-based confidence interval
    alpha = (1 - confidence) / 2
    lower = np.percentile(scores, alpha * 100)
    upper = np.percentile(scores, (1 - alpha) * 100)
    
    return np.mean(scores), lower, upper


def compare_models_statistically(y_test, signal_pred, baseline_pred, 
                                signal_proba=None, baseline_proba=None):
    """
    Perform statistical comparison between signal and baseline models.
    
    Args:
        y_test: True test labels
        signal_pred: Signal model predictions (class labels)
        baseline_pred: Baseline model predictions (class labels)
        signal_proba: Signal model probability predictions (optional)
        baseline_proba: Baseline model probability predictions (optional)
    
    Returns:
        Dictionary with test results
    """
    print(f"\n{'='*70}")
    print("STATISTICAL MODEL COMPARISON")
    print(f"{'='*70}\n")

    # Convert to numpy arrays and validate shapes
    y_test_array = np.asarray(y_test)
    signal_pred_array = np.asarray(signal_pred)
    baseline_pred_array = np.asarray(baseline_pred)

    if y_test_array.shape != signal_pred_array.shape or y_test_array.shape != baseline_pred_array.shape:
        raise ValueError("Predictions and ground-truth labels must have identical shapes for statistical comparison.")

    observed_labels = np.unique(y_test_array)
    unexpected_signal = np.setdiff1d(np.unique(signal_pred_array), observed_labels)
    unexpected_baseline = np.setdiff1d(np.unique(baseline_pred_array), observed_labels)
    if unexpected_signal.size > 0:
        print(f"⚠ Signal model predicted unseen labels: {unexpected_signal}")
    if unexpected_baseline.size > 0:
        print(f"⚠ Baseline model predicted unseen labels: {unexpected_baseline}")

    signal_accuracy = accuracy_score(y_test_array, signal_pred_array)
    baseline_accuracy = accuracy_score(y_test_array, baseline_pred_array)
    print(f"Signal accuracy:   {signal_accuracy:.4f}")
    print(f"Baseline accuracy: {baseline_accuracy:.4f}\n")

    # McNemar's Test for paired model comparison
    print("1. McNemar's Test (Paired Model Comparison)")
    print("-" * 70)

    # Create contingency table
    table = np.zeros((2, 2))
    table[0, 0] = np.sum((signal_pred_array == y_test_array) & (baseline_pred_array == y_test_array))  # Both correct
    table[0, 1] = np.sum((signal_pred_array == y_test_array) & (baseline_pred_array != y_test_array))  # Only signal correct
    table[1, 0] = np.sum((signal_pred_array != y_test_array) & (baseline_pred_array == y_test_array))  # Only baseline correct
    table[1, 1] = np.sum((signal_pred_array != y_test_array) & (baseline_pred_array != y_test_array))  # Both wrong

    print(f"Both models correct:       {table[0, 0]:.0f}")
    print(f"Only signal correct:       {table[0, 1]:.0f}")
    print(f"Only baseline correct:     {table[1, 0]:.0f}")
    print(f"Both models wrong:         {table[1, 1]:.0f}")
    print(f"\nSignal model correct:      {table[0, 0] + table[0, 1]:.0f}")
    print(f"Baseline model correct:    {table[0, 0] + table[1, 0]:.0f}")
    
    # Perform McNemar's test (using chi-square approximation)
    b = table[0, 1]  # Signal correct, baseline wrong
    c = table[1, 0]  # Baseline correct, signal wrong
    
    if b + c > 0:
        mcnemar_stat = (abs(b - c) - 1)**2 / (b + c)
        p_value = 1 - stats.chi2.cdf(mcnemar_stat, df=1)
        
        print(f"\nMcNemar's χ² statistic:    {mcnemar_stat:.4f}")
        print(f"p-value:                   {p_value:.4f}")
        
        if p_value < 0.001:
            print("\n✓ Signal model is HIGHLY SIGNIFICANTLY better (p < 0.001) ***")
        elif p_value < 0.01:
            print("\n✓ Signal model is SIGNIFICANTLY better (p < 0.01) **")
        elif p_value < 0.05:
            print("\n✓ Signal model is SIGNIFICANTLY better (p < 0.05) *")
        else:
            print("\n✗ No significant difference between models (p >= 0.05)")
    else:
        print("\n⚠ Cannot perform McNemar's test: no discordant pairs")
        p_value = np.nan
        mcnemar_stat = np.nan
    
    # Bootstrap confidence intervals for accuracy
    print(f"\n\n2. Bootstrap 95% Confidence Intervals for Accuracy")
    print("-" * 70)
    
    signal_acc_mean, signal_acc_lower, signal_acc_upper = bootstrap_confidence_interval(
        y_test_array, signal_pred_array, 
        lambda y, p: accuracy_score(y, p),
        n_bootstrap=1000
    )
    
    baseline_acc_mean, baseline_acc_lower, baseline_acc_upper = bootstrap_confidence_interval(
        y_test_array, baseline_pred_array,
        lambda y, p: accuracy_score(y, p),
        n_bootstrap=1000
    )
    
    print(f"Signal model:   {signal_acc_mean:.4f} [{signal_acc_lower:.4f}, {signal_acc_upper:.4f}]")
    print(f"Baseline model: {baseline_acc_mean:.4f} [{baseline_acc_lower:.4f}, {baseline_acc_upper:.4f}]")
    print(f"Difference:     {signal_acc_mean - baseline_acc_mean:+.4f}")
    
    # Check if confidence intervals overlap
    if signal_acc_lower > baseline_acc_upper:
        print("\n✓ Confidence intervals do NOT overlap - strong evidence of difference")
    elif signal_acc_upper < baseline_acc_lower:
        print("\n✗ Baseline appears better (confidence intervals do not overlap)")
    else:
        print("\n○ Confidence intervals overlap - weaker evidence of difference")
    
    print(f"\n{'='*70}\n")
    
    return {
        'mcnemar_statistic': mcnemar_stat,
        'mcnemar_p_value': p_value,
        'signal_acc_mean': signal_acc_mean,
        'signal_acc_ci': (signal_acc_lower, signal_acc_upper),
        'baseline_acc_mean': baseline_acc_mean,
        'baseline_acc_ci': (baseline_acc_lower, baseline_acc_upper)
    }


---
# Data Collection & Loading

This stage consolidates the curated CSV exports into runtime data structures. We reload the cached German news scrape and cleaned energy telemetry so that all subsequent feature engineering operates on a controlled snapshot. The code also recomputes baseline targets to ensure label definitions stay in sync with any upstream parameter changes.

In [ ]:
# ============================================================================
# STAGE 1 · DATA INGESTION & BASELINE FEATURES
# ============================================================================

def run_ingestion_stage(
    news_path: str = "german_news_v1.csv",
    energy_path: str = "energy_data.csv",
    min_timestamp: str = "2025-01-01",
    news_sample: int | None = None,
):
    with StageProfiler("Stage 1 — Ingestion & Baseline Features", device_config):
        # --- Load & normalize news feed ---
        news_df = pd.read_csv(news_path)
        news_df['publishedAt'] = pd.to_datetime(news_df['publishedAt'])
        news_df['publishedAt'] = news_df['publishedAt'].dt.tz_localize(None)
        news_df = news_df.set_index('publishedAt').sort_index()
        if min_timestamp:
            news_df = news_df[news_df.index >= min_timestamp]
        if news_sample is not None:
            news_df = news_df.sample(news_sample, random_state=RANDOM_STATE)

        # --- Load energy telemetry ---
        energy_df = pd.read_csv(energy_path)
        energy_df['Timestamp'] = pd.to_datetime(energy_df['Timestamp']) - pd.Timedelta(hours=1)
        energy_df = energy_df.set_index('Timestamp').sort_index()
        energy_df = energy_df.drop(columns=['Nuclear'])
        energy_df = energy_df.dropna()
        if min_timestamp:
            energy_df = energy_df[energy_df.index >= f"{min_timestamp} 00:00:00"]

        # --- Baseline feature computation ---
        energy_df['real_spread_abs'] = energy_df['Spot Price'] - energy_df['Day Ahead Auction']
        energy_df = energy_df.drop(columns=['Non-Renewable', 'Renewable'])

        spread_diff = energy_df['real_spread_abs']
        spread_target = np.sign(spread_diff).astype(int)
        neutral_mask = spread_diff.abs() <= SPREAD_TARGET_DEADBAND
        spread_target[neutral_mask] = 0
        energy_df['spread_target'] = spread_target

        master_df = energy_df.copy()
        master_df['real_spread_abs_shift_24'] = master_df['real_spread_abs'].shift(-FORECAST_HORIZON_HOURS)
        master_df['spread_target_shift_24'] = master_df['spread_target'].shift(-FORECAST_HORIZON_HOURS)
        master_df['price_lag_24'] = master_df['Spot Price'].shift(FORECAST_HORIZON_HOURS)
        master_df['price_lag_168'] = master_df['Spot Price'].shift(HOURS_PER_WEEK)
        master_df['load_lag_24'] = master_df['Load'].shift(FORECAST_HORIZON_HOURS)
        master_df['load_lag_168'] = master_df['Load'].shift(HOURS_PER_WEEK)
        master_df['hour'] = master_df.index.hour
        master_df['week_of_year'] = master_df.index.isocalendar().week
        master_df['month'] = master_df.index.month
        master_df['day_of_week'] = master_df.index.dayofweek
        master_df['day_of_year'] = master_df.index.dayofyear
        master_df = master_df.dropna()

        print(f"News shape after filters: {news_df.shape}")
        print(f"Energy telemetry shape: {energy_df.shape}")
        print(f"Baseline feature frame: {master_df.shape}")

        return {
            "news_df": news_df,
            "energy_df": energy_df,
            "master_df": master_df,
        }


ingestion_outputs = run_ingestion_stage()
news_df = ingestion_outputs['news_df']
energy_df = ingestion_outputs['energy_df']
master_df = ingestion_outputs['master_df']


### Optional: Streaming Data Pipeline

For larger reruns we can materialise parquet caches via the GPU-aware streaming pipeline.

**What is a Streaming Pipeline?**
Instead of loading the entire dataset into RAM at once (which may exceed memory limits for multi-year news archives), a streaming pipeline processes data in chunks:
1. Read 10,000 rows
2. Apply transformations (filtering, embedding generation)
3. Write to disk
4. Repeat

**What is Parquet?**
Parquet is a columnar storage format optimized for analytics:
- **Compression:** 5-10x smaller than CSV (fewer disk I/O operations)
- **Column Pruning:** Read only the columns you need (e.g., skip article text if you only need timestamps)
- **Schema Enforcement:** Guarantees consistent data types (no "date parsed as string" bugs)

**GPU-Aware Streaming:**
When generating embeddings (next section), the pipeline batches articles to maximize GPU throughput:
- CPU: Load and tokenize batch
- GPU: Generate embeddings
- CPU: Write to parquet
This overlaps I/O and computation for 2-3x speedups vs sequential processing.

**When to Use This:**
- Initial data ingestion from raw scrapes
- Regenerating features after parameter changes
- Scaling beyond laptop RAM (e.g., processing 5M+ articles)

In [ ]:
from gpu_pipeline.data_pipeline import DataPipelineConfig, StreamingDataPipeline

pipeline_config = DataPipelineConfig(
    energy_source=Path("energy_data.csv"),
    news_source=Path("german_news_v1.csv"),
    artifact_dir=Path(".cache/data_pipeline"),
    cache_max_bytes=32 * 1024**3,  # keep local footprint under 32 GB
    remote_cache_uri=os.getenv("AREP_REMOTE_CACHE"),
)
streaming_pipeline = StreamingDataPipeline(pipeline_config)

# Generate parquet caches lazily and prune stale artefacts
streaming_pipeline.prepare_artifacts()

# Example: iterate over the first 2 daily energy batches without materialising the full dataset
for idx, energy_batch in enumerate(streaming_pipeline.iter_energy(batches=2)):
    print(f"Energy batch {idx+1}: {energy_batch.shape}")

# Example: news streaming (limit to first batch for inspection)
for news_batch in streaming_pipeline.iter_news(batches=1):
    display(news_batch.head())
    break


### Inspect News Snapshot

A quick peek at the fully loaded news frame confirms the ingestion filters behaved as expected before we commit to heavy NLP processing.

**What to Check:**
1. **Date Range:** Are articles within the expected timeframe? (e.g., 2019-2023 for German energy market)
2. **Language:** All German-language articles? (English or other languages may slip through web scraping)
3. **Source Distribution:** Are major outlets represented? (e.g., Handelsblatt, FAZ, Reuters)
4. **Missing Data:** Any null titles or descriptions that would break zero-shot classification?

**Why Spot-Check:**
Even with the full dataset in memory, inspecting a handful of rows catches early issues:
- **Encoding Problems:** Garbled characters from incorrect UTF-8 handling
- **Duplicate Articles:** Same article scraped multiple times with different timestamps
- **Irrelevant Content:** Sports or celebrity news that passed keyword filters but isn't energy-related

**Example Validation:**
```python
# Check for unexpected languages
df['title'].str.contains('solar|wind|nuclear').mean()  # Should be high

# Check for duplicates
df.duplicated(subset=['title', 'published_date']).sum()  # Should be ~0
```

If these checks pass, we proceed confidently to the expensive NLP stages.

In [ ]:
display(news_df.head())
print(f"News timeframe: {news_df.index.min()} → {news_df.index.max()}")
print(f"News articles loaded: {len(news_df)}")

## Energy Data

We immediately sanity-check the raw energy frame to understand coverage issues (e.g., missing nuclear data) before deriving features.

**What is the Energy Dataset?**
This dataframe contains hourly telemetry from the German power grid:
- **Prices:** Day-ahead auction price (EUR/MWh), intraday continuous price
- **Supply:** Generation from wind, solar, nuclear, coal, gas (MW)
- **Demand:** Total load, cross-border imports/exports (MW)
- **Spreads:** Our target variable—difference between forecasted and realized price

**Common Data Quality Issues:**
1. **Missing Nuclear Data:** Germany phased out nuclear reactors 2011-2023, so later years have NaNs
2. **Timezone Shifts:** Data may be in UTC but German markets use CET/CEST (1-2 hour offset)
3. **Outliers:** Occasional spikes from data provider errors or extreme events (e.g., French nuclear outages causing import surges)

**Why Sanity-Check Now:**
If nuclear generation has 80% missing values, we need to decide:
- Drop the feature entirely?
- Impute with zero (assuming decommissioned)?
- Create a binary "nuclear_active" flag?

Identifying this early prevents downstream model failures like "XGBoost crashes on NaN input" or "feature importance shows nuclear as top predictor but it's actually all imputed zeros".

In [ ]:
energy_df.tail(5)

### Inspect Energy Coverage

We sanity-check the telemetry horizon and first rows to ensure timezone adjustments and column drops produced a coherent baseline dataset.

**Checks to Perform:**
1. **Date Range Coverage:**
   ```python
   print(f"Start: {df.index.min()}")  # Should match news dataset start
   print(f"End: {df.index.max()}")    # Should cover test period
   ```
   Gaps indicate missing data from API outages or provider issues.

2. **Timezone Consistency:**
   ```python
   df.index.tz  # Should be 'Europe/Berlin' or 'UTC' consistently
   ```
   Mixing timezones causes off-by-one-hour errors when merging with news.

3. **Column Completeness:**
   ```python
   df.isnull().sum()  # Count missing values per column
   ```
   High missingness (>50%) suggests a column is unreliable and should be excluded.

4. **Value Ranges:**
   ```python
   df.describe()  # Check min/max for outliers
   ```
   Example: Negative prices occur occasionally (oversupply), but values like -9999 signal data errors.

**What Happened in Pre-Processing:**
- Dropped columns with >80% missing data (e.g., decommissioned nuclear plants)
- Converted timestamps from UTC to Berlin time (aligning with market participants' decision-making timezone)
- Forward-filled small gaps (<2 hours) to avoid fragmenting the time series

If these checks pass, the energy data is ready for feature engineering (lags, rolling averages, spreads).

In [ ]:
print(f"Energy telemetry timeframe: {energy_df.index.min()} → {energy_df.index.max()}")
energy_df.head(5)

---
# Feature Engineering

The next blocks enrich the baseline dataset with time-aware signals drawn from both market telemetry and news analytics. Feature engineering is the process of transforming raw data into predictive inputs that machine learning models can understand and learn from effectively.

**Why Feature Engineering Matters:**
In time-series forecasting, raw data (prices, timestamps, article text) often lacks the structure models need to discover patterns. Feature engineering explicitly encodes domain knowledge:
- **Temporal Patterns:** Hour-of-day captures daily cycles (night vs day demand)
- **Historical Context:** Lags and moving averages provide memory of recent events
- **Semantic Signals:** News topics and embeddings translate text into numerical vectors

**Our Feature Strategy:**
We'll create three feature families, each with a clear hypothesis about what drives German power spreads:

1. **Baseline Energy Features (Cells 34-38):**
   - Price lags and rolling statistics (momentum, volatility)
   - Supply/demand metrics (renewable share, imports/exports)
   - Temporal encodings (hour, day, month)
   - **Hypothesis:** Recent prices and grid state predict near-term spreads

2. **News Topic Features (Cells 40-44):**
   - Zero-shot classification into market-relevant categories
   - Time-decayed aggregation (recent news weighs more)
   - **Hypothesis:** Supply shocks and weather events mentioned in news correlate with price volatility

3. **News Embedding Features (Cells 46-60):**
   - Dense semantic vectors capture nuances beyond discrete topics
   - Dimensionality reduction (UMAP) for computational efficiency
   - **Hypothesis:** Articles with similar semantic content precede similar market movements

Each feature family will be profiled, validated, and merged into a master dataset for model training. This modular approach allows us to later perform ablation studies (e.g., "Does dropping news features hurt performance?") to justify each component's value.

In [ ]:
if 'telemetry_logger' in globals():
    telemetry_logger.metadata.update({
        'news_rows': len(news_df),
        'energy_rows': len(energy_df),
    })


### Check Target Balance

Validating the class distribution early tells us whether the neutral band width creates imbalance that later models must address.

**What is Class Imbalance?**
Our target variable has three classes:
- **Long:** Price spread widened significantly (buy signal)
- **Neutral:** Price spread stayed within ±threshold (hold signal)
- **Short:** Price spread narrowed significantly (sell signal)

If the distribution is heavily skewed (e.g., 70% Neutral, 15% Long, 15% Short), models may learn to always predict "Neutral" and still achieve 70% accuracy—but earn no trading profit.

**Why It Matters:**
- **Model Bias:** XGBoost and LightGBM default to optimizing overall accuracy, which favors the majority class
- **Evaluation Metrics:** Accuracy becomes misleading; we need precision/recall per class or F1-score
- **Resampling Strategies:** Severe imbalance may require SMOTE (synthetic oversampling) or class-weighted loss functions

**Typical Target:**
For profitable trading, aim for roughly balanced classes (e.g., 40% Neutral, 30% Long, 30% Short). This is controlled by the "neutral band width" parameter (e.g., ±1 EUR/MWh).

**Check Output:**
```python
target.value_counts(normalize=True)
# Neutral: 0.42
# Long: 0.31
# Short: 0.27  ← Reasonably balanced
```

If one class is <10%, consider adjusting the neutral band or using stratified sampling in cross-validation.

In [ ]:
print("Baseline target distribution:")
print(master_df['spread_target'].value_counts(normalize=True).round(3))


### Preview Baseline Features

Reviewing the head of `master_df` verifies the engineered shifts/lag features exist before we layer on news-derived signals.

**What Are Baseline Features?**
These are time-series features derived purely from energy market data, without news:
1. **Lagged Prices:** `price_lag_1h`, `price_lag_24h` (yesterday's price at same hour)
2. **Rolling Averages:** `price_ma_7d` (7-day moving average smooths out daily volatility)
3. **Spread Features:** `actual_price - forecasted_price` (the basis for our target)
4. **Supply/Demand Ratios:** `renewable_share = (wind + solar) / total_generation`

**Why Preview Now:**
- **Column Names:** Confirm features are named consistently (e.g., `_lag_1h` vs `_1h_lag`)
- **No NaNs in Critical Rows:** First 24-48 rows may have NaNs from lags; ensure they're dropped in training splits
- **Value Sanity:** Negative prices or impossibly high ratios (>1.0 for renewable share) indicate bugs

**Example Output:**
```
timestamp           price_lag_1h  price_ma_7d  renewable_share  target
2020-01-15 10:00    45.2          48.1         0.38            Neutral
2020-01-15 11:00    47.3          48.0         0.41            Long
```

If this looks correct, we're ready to merge news features (topics, embeddings) in subsequent cells.

In [ ]:
master_df.head()

### Confirm Feature Inventory

We list the current feature columns so that downstream selection logic can reference an authoritative set without hidden assumptions.

**Why Explicitly List Features?**
In a large notebook, it's easy to lose track of which features exist after multiple transformations. An explicit inventory:
- **Prevents Bugs:** "I thought I created `price_momentum` but it's not in the list" → catch the error now
- **Documents Feature Engineering:** Stakeholders can see exactly what inputs the model receives
- **Enables Feature Selection:** Later, we can programmatically select subsets (e.g., "all `_lag` features" vs "news-derived features")

**Typical Categories:**
1. **Time Features:** `hour_of_day`, `day_of_week`, `is_weekend` (capture daily/weekly patterns)
2. **Price Lags:** Historical prices at various offsets (1h, 3h, 24h)
3. **Rolling Statistics:** Moving averages, standard deviations (smooth noise)
4. **Supply/Demand:** Generation by source, total load, import/export
5. **Spread Metrics:** Forecast errors, volatility measures

**Example Output:**
```python
print(f"Total features: {len(master_df.columns)}")
print(f"Feature categories: {set([col.split('_')[0] for col in master_df.columns])}")
# Output: Total features: 47
#         Categories: {'price', 'load', 'wind', 'solar', 'hour', 'spread'}
```

**Next Steps:**
After news features are added (topics, embeddings), we'll re-run this check to confirm the final feature count.

In [ ]:
print("Feature columns prepared:")
print(master_df.columns.tolist())
master_df.head(25)

## Topic Taxonomy Design

We define the candidate topic labels and hypotheses that guide zero-shot classification. The list reflects the drivers we expect to move German power spreads.

**What is Zero-Shot Classification?**
Unlike traditional ML (train a model on labeled examples), zero-shot uses a pre-trained language model to classify text into categories it has never explicitly seen:
- **Input:** Article text + list of candidate labels
- **Output:** Probability distribution over labels (e.g., 70% "supply_shock", 20% "demand", 10% "policy")

**Why Zero-Shot?**
- **No Labeled Data Needed:** We don't have 10K manually tagged energy articles
- **Flexible Taxonomy:** Easy to add/remove categories without retraining
- **Multilingual:** Pre-trained models like `Sahajtomar/German_Zeroshot` handle German text natively

**Chosen Topics and Rationale:**
1. **Supply Shock:** Unplanned nuclear outages, pipeline disruptions → drives prices up
2. **Demand Surge:** Cold snaps, industrial production spikes → increases load
3. **Weather Extreme:** Windless weeks, solar-blocking storms → reduces renewable generation
4. **Policy/Regulation:** Carbon tax changes, grid fee adjustments → shifts market structure
5. **Financial Stress:** Utility bankruptcies, margin calls → volatility
6. **Other:** Catch-all for articles mentioning energy but not fitting above (e.g., "new wind farm" announcements with long lead times)

**Hypothesis:**
Articles tagged "supply_shock" or "weather_extreme" in the past 6-24 hours should correlate with wider price spreads (our target). We'll validate this during feature importance analysis.

In [ ]:
# TODO: Fine tune these topics

candidate_labels = [
    # Energy consumption
    "der Strom- oder Energieverbrauch steigt deutlich",
    "der Strom- oder Energieverbrauch sinkt deutlich",

    # Energy production / generation availability
    "die Erzeugung oder Verfügbarkeit von Strom oder Energie steigt",
    "die Erzeugung oder Verfügbarkeit von Strom oder Energie sinkt",

    # Commodity prices (gas/coal/oil)
    "die Preise für Erdgas, Kohle oder Öl steigen stark",
    "die Preise für Erdgas, Kohle oder Öl fallen stark",

    # Geopolitik und Versorgung
    "es gibt zunehmende geopolitische Spannungen oder Lieferengpässe bei Energie",
    "die geopolitischen Spannungen und Versorgungsprobleme gehen zurück",

    # Auswirkungen von Wetter in Deutschland auf Strompreise
    "in Deutschland gibt es kaltes Wetter, wenig Wind oder wenig Sonne mit möglichen Auswirkungen auf die Strompreise",
    "in Deutschland gibt es mildes oder warmes Wetter, viel Wind oder viel Sonne mit möglichen Auswirkungen auf die Strompreise",

    # Finanzmärkte
    "an den Finanzmärkten herrscht Instabilität oder Turbulenz",
    "an den Finanzmärkten herrscht Stabilität oder Beruhigung",

    # Catch-all
    "der Text hat keinen Bezug zu Energiepreisen, Wetter oder Finanzmärkten"
]

hypothesis_template = "Dieser Text legt nahe, dass {}."

## Zero-Shot Classification

We rely on a zero-shot classifier to tag each article with the most plausible topic. The intent is to capture directional narratives (supply shocks, demand swings, weather extremes, financial stress) without training a bespoke model.

**Model Choice: `Sahajtomar/German_Zeroshot`**
- **Language-Specific:** Fine-tuned on German text, avoiding translation overhead and preserving nuance
- **Lightweight:** ~400MB model fits on modest GPUs or runs acceptably on CPU
- **Pre-Trained on NLI:** Uses natural language inference (premise-hypothesis pairs) to assess semantic similarity

**How Zero-Shot Works:**
1. **Tokenization:** Convert article text to token IDs (subword units)
2. **Embedding:** Pass through transformer layers to get contextual representations
3. **Hypothesis Testing:** For each candidate label (e.g., "supply shock"), frame it as a hypothesis: "This article discusses supply shocks."
4. **Scoring:** Compute entailment probability (how likely the article entails the hypothesis)
5. **Selection:** Assign the label with the highest probability

**Handling "Other" Articles:**
Articles tagged "Other" with low confidence (<0.5) are reprocessed with full descriptions instead of just titles, giving the model more context. This reduces noise in the catch-all category.

**Performance Trade-Offs:**
- **Speed:** ~50-200 articles/sec on GPU, ~5-10 articles/sec on CPU
- **Accuracy:** Not as precise as supervised models (would require labeled data), but sufficient for feature engineering where aggregate topic trends matter more than individual article precision

**Output:**
Each article gets a `topic` column (e.g., "supply_shock") and a `topic_score` (0-1 confidence). Low-confidence classifications can be filtered or weighted differently in downstream aggregations.

In [ ]:
# ============================================================================
# STAGE 2 · NLP TOPIC CLASSIFICATION & EMBEDDINGS PREP
# ============================================================================


def run_embedding_stage(
    news_df: pd.DataFrame,
    candidate_labels: list[str],
    hypothesis_template: str,
    batch_size: int | None = None,
    model_name: str = "Sahajtomar/German_Zeroshot",
):
    news_df = news_df.copy()
    with StageProfiler("Stage 2 — Topic attribution", device_config):
        import os

        os.environ["TOKENIZERS_PARALLELISM"] = "true"
        torch.backends.cudnn.benchmark = True

        effective_batch_size = batch_size or device_config.get("optimal_batch_size", 32)
        hf_device = resolve_hf_device(PRIMARY_DEVICE)
        torch_dtype = torch.float16 if hf_device in (0, "cuda") else torch.float32

        pipeline_kwargs = {
            "model": model_name,
            "batch_size": effective_batch_size,
            "torch_dtype": torch_dtype,
        }
        if hf_device != -1:
            if ACCELERATE_AVAILABLE:
                pipeline_kwargs["device"] = hf_device
            else:
                print("⚠️ Hugging Face accelerate is not installed; zero-shot classifier will run on CPU.")
                pipeline_kwargs["torch_dtype"] = torch.float32

        def instantiate_zero_shot(**kwargs):
            return pipeline("zero-shot-classification", **kwargs)

        resolved_hf_device = pipeline_kwargs.get("device", hf_device)
        if not ACCELERATE_AVAILABLE and hf_device != -1:
            resolved_hf_device = "cpu"
        try:
            classifier = instantiate_zero_shot(**pipeline_kwargs)
        except ValueError as exc:
            message = str(exc).lower()
            if "cannot be moved to a specific device" in message and pipeline_kwargs.pop("device", None) is not None:
                print("⚠️ accelerate-managed model detected; retrying without explicit device placement.")
                classifier = instantiate_zero_shot(**pipeline_kwargs)
                resolved_hf_device = "accelerate-auto"
            else:
                raise
        except RuntimeError as exc:
            message = str(exc)
            if hf_device == "mps" and "mps" in message.lower():
                print("⚠️ MPS pipeline creation failed; falling back to CPU.")
                pipeline_kwargs.pop("device", None)
                pipeline_kwargs["torch_dtype"] = torch.float32
                classifier = instantiate_zero_shot(**pipeline_kwargs)
                resolved_hf_device = "cpu"
            else:
                raise

        if "device" not in pipeline_kwargs and resolved_hf_device != "accelerate-auto":
            candidate_device = getattr(classifier, "device", None)
            if candidate_device is not None:
                resolved_hf_device = candidate_device

        resolved_hf_device_repr = resolved_hf_device
        if isinstance(resolved_hf_device_repr, torch.device):
            device_type = resolved_hf_device_repr.type
            index = resolved_hf_device_repr.index
            resolved_hf_device_repr = device_type if index is None else f"{device_type}:{index}"

        def classify_batch(texts, labels, hypothesis_template, show_progress=True):
            valid_pairs = [(idx, text) for idx, text in enumerate(texts) if pd.notna(text) and text.strip()]
            if not valid_pairs:
                return {}, {}

            _, filtered_texts = zip(*valid_pairs)
            if show_progress:
                print(
                    f"Processing {len(filtered_texts)} texts with batch_size={effective_batch_size}"
                )

            results = classifier(
                list(filtered_texts),
                labels,
                hypothesis_template=hypothesis_template,
                multi_label=False,
            )
            if isinstance(results, dict):
                results = [results]

            classifications = {}
            scores = {}
            for (idx, _), result in zip(valid_pairs, results):
                classifications[idx] = result['labels'][0]
                scores[idx] = float(result['scores'][0])

            for idx in range(len(texts)):
                classifications.setdefault(idx, None)
                scores.setdefault(idx, 0.0)

            return classifications, scores

        titles = news_df['title'].tolist()
        classifications_dict, scores_dict = classify_batch(
            titles,
            candidate_labels,
            hypothesis_template,
            show_progress=True,
        )

        news_df['classification'] = [classifications_dict[i] for i in range(len(news_df))]
        news_df['classification_score'] = [scores_dict[i] for i in range(len(news_df))]

        other_mask = news_df['classification'] == "other (not related to these energy price drivers)"
        num_other = other_mask.sum()

        if num_other > 0:
            other_indices = news_df[other_mask].index
            descriptions = news_df.loc[other_indices, 'description'].tolist()
            other_classifications_dict, other_scores_dict = classify_batch(
                descriptions,
                candidate_labels,
                hypothesis_template,
                show_progress=True,
            )
            for i, idx in enumerate(other_indices):
                news_df.loc[idx, 'classification'] = other_classifications_dict[i]
                news_df.loc[idx, 'classification_score'] = other_scores_dict[i]

        final_other = (
            news_df['classification']
            == "es ist nichts mit Energiepreisen, Wetter oder Finanzmärkten zu tun hat"
        ).sum()

        print(f"Classification completed: {len(news_df)} articles processed")
        print(f"Articles classified as 'other': {final_other} ({final_other / len(news_df) * 100:.1f}%)")
        print("\nClassification distribution:")
        print(news_df['classification'].value_counts())
        print(f"\nAverage score: {news_df['classification_score'].mean():.3f}")
        print(f"Median score: {news_df['classification_score'].median():.3f}")

        return {
            "news_df": news_df,
            "classifier": classifier,
            "batch_size": effective_batch_size,
            "hf_device": resolved_hf_device_repr,
        }


embedding_outputs = run_embedding_stage(news_df, candidate_labels, hypothesis_template)
news_df = embedding_outputs['news_df']
zero_shot_classifier = embedding_outputs['classifier']
embedding_batch_size = embedding_outputs['batch_size']

### Inspect Classified News

Viewing a few rows after topic attribution helps confirm the zero-shot classifier is producing plausible labels and scores.

**What to Validate:**
1. **Topic Plausibility:** Do assigned labels match article content?
   ```python
   # Example: Article titled "Gaskrise verschärft sich" (Gas crisis worsens)
   # Should be labeled "supply_shock", not "other"
   ```

2. **Score Distribution:** Are confidence scores reasonable?
   ```python
   df['topic_score'].describe()
   # Mean: 0.65-0.75 is typical for zero-shot (not as confident as supervised)
   # Min: <0.3 suggests classifier is guessing randomly
   ```

3. **"Other" Category Size:** Is the catch-all too large?
   ```python
   (df['topic'] == 'other').mean()
   # Target: <30%; if >50%, taxonomy may be too narrow
   ```

**Red Flags:**
- **All Articles Labeled "Other":** Model failed to load or candidate labels don't match article language
- **Uniform Scores (~0.33 for 3 classes):** Random guessing; check that article text isn't empty or truncated
- **Suspicious Patterns:** E.g., all nighttime articles labeled "demand_surge" due to timestamp leakage in text

**Example Validation Code:**
```python
# Manually inspect high-confidence mismatches
sample = df[df['topic_score'] > 0.8].sample(10)
for idx, row in sample.iterrows():
    print(f"Topic: {row['topic']} | Score: {row['topic_score']:.2f}")
    print(f"Title: {row['title']}")
    print()
```

If classifications look reasonable, proceed to embedding generation. If many are nonsensical, revisit the topic taxonomy or try a different zero-shot model.

In [ ]:
news_df.head(5)

## News Embeddings Features

Beyond discrete topics we also propagate dense semantic embeddings. Embeddings are high-dimensional vectors (e.g., 768 numbers) that capture the semantic meaning of text.

**What Are Embeddings?**
Imagine projecting every article into a 768-dimensional space where semantically similar articles are close together:
- "Nuclear plant shuts down" and "Reactor goes offline" → nearby vectors
- "New solar farm opens" → far from nuclear articles

**Why Use Embeddings Alongside Topics?**
- **Topics Are Coarse:** "Supply shock" lumps together nuclear outages, pipeline breaks, and coal plant failures—embeddings preserve nuances
- **Catch-All Coverage:** Articles labeled "Other" by zero-shot still carry signal in embedding space (e.g., "new wind farm" may correlate with future supply increases)
- **Nonlinear Relationships:** XGBoost/LightGBM can discover complex patterns (e.g., "articles similar to 2019 cold snap cluster" predict demand surges)

**Model Choice: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**
- **Multilingual:** Supports German (and 50+ other languages)
- **Compact:** 768 dimensions (vs 1024+ for larger models)
- **Fast:** ~1000 articles/sec on GPU

**Caching and Device Placement:**
The upcoming function:
1. Checks if embeddings are already cached on disk (parquet file)
2. If not, batches articles to GPU for encoding
3. Saves results to avoid recomputation on notebook restart

**Memory Considerations:**
100K articles × 768 dimensions × 4 bytes/float = 307 MB. For larger datasets (1M+ articles), we'll compress with UMAP after generation (next section).

In [ ]:
def compute_embeddings(
    news_df: pd.DataFrame,
    model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2',
    batch_size: int | None = None,
    show_progress: bool = True,
    cache_dir: Path | None = None,
) -> pd.DataFrame:
    """Attach SentenceTransformer embeddings to the news dataframe with GPU-first caching."""
    import hashlib
    import json
    import pyarrow as pa
    import pyarrow.parquet as pq

    def _title_checksum(frame: pd.DataFrame) -> str:
        digest = hashlib.sha1()
        for title in frame['title'].fillna('').astype(str):
            digest.update(title.encode('utf-8'))
        return digest.hexdigest()

    cache_root = cache_dir
    if cache_root is None:
        cache_root = Path('.cache/embeddings')
        if 'pipeline_config' in globals():
            try:
                cache_root = Path(globals()['pipeline_config'].artifact_dir) / 'embeddings'
            except AttributeError:
                pass
    cache_root.mkdir(parents=True, exist_ok=True)
    cache_path = cache_root / 'news_embeddings.parquet'
    manifest_path = cache_root / 'manifest.json'

    checksum = _title_checksum(news_df)
    if cache_path.exists() and manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
            if manifest.get('checksum') == checksum:
                embeddings_table = pq.read_table(cache_path)
                vectors = np.array([list_val for list_val in embeddings_table['embedding'].to_pylist()], dtype=np.float32)
                enriched_df = news_df.copy()
                enriched_df['embedding'] = [vec for vec in vectors]
                print(f"Loaded embeddings from cache ({cache_path})")
                return enriched_df
        except Exception as cache_exc:
            print(f"Cache read skipped: {cache_exc}")

    with StageProfiler("Stage 2 — SentenceTransformer embeddings", device_config):
        hf_device = PRIMARY_DEVICE if PRIMARY_DEVICE in {"cuda", "mps"} else "cpu"
        effective_batch_size = batch_size or embedding_outputs.get('batch_size', device_config.get('optimal_batch_size', 32))

        model = SentenceTransformer(model_name, device=hf_device)
        if hf_device == "cuda":
            try:
                model = model.half()
                print("Model converted to fp16 (half precision)")
            except Exception as exc:
                print(f"fp16 conversion skipped: {exc}")

        texts = [
            (news_df.iloc[idx]['title'] if pd.notna(news_df.iloc[idx]['title']) else '').strip()
            for idx in range(len(news_df))
        ]
        valid_pairs = [(i, text) for i, text in enumerate(texts) if text]
        if not valid_pairs:
            print("Warning: No valid texts found for embedding!")
            return news_df

        _, valid_texts = zip(*valid_pairs)
        print(f"Using batch_size={effective_batch_size} for embedding computation on device={hf_device}")

        try:
            embeddings_array = model.encode(
                list(valid_texts),
                batch_size=effective_batch_size,
                convert_to_numpy=True,
                show_progress_bar=show_progress,
                normalize_embeddings=False,
            )
        except Exception as primary_exc:
            print(f"Primary embedding pass failed ({primary_exc}); retrying with batch_size=32")
            embeddings_array = model.encode(
                list(valid_texts),
                batch_size=32,
                convert_to_numpy=True,
                show_progress_bar=show_progress,
                normalize_embeddings=False,
            )

        full_embeddings = np.full((len(texts), embeddings_array.shape[1]), np.nan, dtype=np.float32)
        for idx, (row_idx, _) in enumerate(valid_pairs):
            full_embeddings[row_idx] = embeddings_array[idx]

        if hf_device == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        elif hf_device == "mps" and hasattr(torch, "mps"):
            try:
                torch.mps.empty_cache()
            except AttributeError:
                pass

    enriched_df = news_df.copy()
    enriched_df['embedding'] = [emb for emb in full_embeddings]

    try:
        flattened = full_embeddings.reshape(-1)
        list_array = pa.FixedSizeListArray.from_arrays(pa.array(flattened, type=pa.float32()), full_embeddings.shape[1])
        pq.write_table(pa.Table.from_arrays([list_array], names=['embedding']), cache_path)
        manifest_path.write_text(json.dumps({'checksum': checksum, 'embedding_dim': int(full_embeddings.shape[1])}))
    except Exception as write_exc:
        print(f"Embedding cache write skipped: {write_exc}")

    print(f"Embeddings computed: shape {full_embeddings.shape}")
    return enriched_df


news_df = compute_embeddings(news_df, show_progress=True)


### Validate Embedding Attachment

We examine the dataframe shape and head after embedding enrichment to ensure caching and GPU placement yielded complete vectors.

**Checks to Perform:**
1. **Column Existence:** Confirm `embedding` column was added
   ```python
   assert 'embedding' in df.columns
   ```

2. **Vector Dimensions:** Each embedding should be a 768-element array
   ```python
   sample_embedding = df['embedding'].iloc[0]
   assert len(sample_embedding) == 768
   ```

3. **No Missing Embeddings:** Check for NaNs or empty arrays
   ```python
   assert df['embedding'].isna().sum() == 0
   assert all(len(emb) == 768 for emb in df['embedding'])
   ```

4. **Sanity Check Values:** Embeddings are typically normalized (length ≈ 1)
   ```python
   import numpy as np
   norms = df['embedding'].apply(lambda x: np.linalg.norm(x))
   print(f"Mean norm: {norms.mean():.2f}")  # Should be ~1.0
   ```

**Common Failures:**
- **Empty Arrays:** Model failed to load; check Hugging Face cache directory
- **All Zeros:** GPU ran out of memory; reduce batch size
- **Dimension Mismatch:** Downloaded wrong model variant (base vs large)

**If Validation Passes:**
We proceed to time-decay aggregation, where these embeddings will be averaged across hourly windows with exponential weighting.

In [ ]:
print(news_df.shape)
print(news_df.head(5))

## Time-Decayed Topic Aggregation

We aggregate topic counts with exponential decay to emphasise the most recent narratives. This transformation mirrors how market attention fades, letting the model balance recency with enough history to stay stable.

**Why Time Decay?**
In energy markets, yesterday's news about a nuclear outage still matters today, but last week's less so. Exponential decay models this:
```
Weight = λ^(hours_ago)
```
where λ (lambda) is the decay rate (e.g., 0.95).

**Example with λ=0.95:**
- Article 1 hour ago: weight = 0.95^1 = 0.95
- Article 24 hours ago: weight = 0.95^24 = 0.29
- Article 168 hours ago (7 days): weight = 0.95^168 = 0.00014 (negligible)

**Aggregation Process:**
For each hour and each topic, sum the decayed counts:
```python
for timestamp in hourly_index:
    for topic in ['supply_shock', 'demand', ...]:
        # Filter articles before timestamp with this topic
        relevant_articles = news[(news.published <= timestamp) & (news.topic == topic)]

        # Calculate decay weights
        hours_ago = (timestamp - relevant_articles.published).dt.total_seconds() / 3600
        weights = lambda_value ** hours_ago

        # Sum weighted counts
        features[timestamp][f'{topic}_decayed'] = weights.sum()
```

**Hyperparameter Search:**
We'll grid-search λ values (e.g., 0.85, 0.90, 0.95, 0.98) and time horizons (12h, 24h, 72h) to find the optimal recency-vs-history tradeoff. Ridge regression (fast linear model) will screen candidates before expensive XGBoost tuning.

**Output:**
A matrix with shape `[num_hours, num_topics]` where each cell is the decayed topic count at that timestamp.

In [ ]:
def compute_time_decayed_topic_counts(news_df, master_df, lookback_window=DEFAULT_LOOKBACK_WINDOW, decay_lambda=DEFAULT_DECAY_LAMBDA, verbose=True):
    """
    Compute time-decayed weighted counts for each topic.

    For each timestamp in master_df, compute the weighted average count of articles
    published within the lookback_window, using exponential time decay.

    Weight formula: weight = e^(-lambda * hours_since_publication)

    Parameters:
        news_df: DataFrame with datetime index (publishedAt), with columns ['classification', ...]
        master_df: DataFrame with datetime index representing target timestamps
        lookback_window: Number of hours to look back (default: 336h = 2 weeks)
        decay_lambda: Decay rate parameter (default: 0.05)
        verbose: Whether to print progress messages (default: True)

    Returns:
        pd.DataFrame: DataFrame with datetime index and columns for each topic's weighted count
    """

    # Make a copy to avoid modifying original
    td_topic_news_df = news_df.copy()
    # Ensure index is datetime and reset timezone if needed
    if not isinstance(td_topic_news_df.index, pd.DatetimeIndex):
        td_topic_news_df.index = pd.to_datetime(td_topic_news_df.index)
    if td_topic_news_df.index.tz is not None:
        td_topic_news_df.index = td_topic_news_df.index.tz_localize(None)

    # Get unique topics (excluding NaN)
    topics = td_topic_news_df['classification'].dropna().unique()

    # Create output DataFrame with same index as master_df
    td_topics_df = pd.DataFrame(index=master_df.index)

    if verbose:
        print(f"Computing time-decayed counts for {len(td_topics_df)} timestamps and {len(topics)} topics")
        print(f"Lookback window: {lookback_window}h, decay lambda: {decay_lambda}")
        print("Using vectorized computation for improved performance...")

    # Prepare vectorized data
    timestamps = td_topics_df.index.values
    article_times = td_topic_news_df.index.values  # publishedAt as index
    article_topics = td_topic_news_df['classification'].values

    # Pre-filter valid articles (those with valid topics)
    valid_mask = pd.notna(article_topics)
    article_times_valid = article_times[valid_mask]
    article_topics_valid = article_topics[valid_mask]

    # Topic to col idx
    topic_to_idx = {topic: idx for idx, topic in enumerate(topics)}
    weighted_counts_array = np.zeros((len(timestamps), len(topics)))

    if verbose:
        print(f"Processing {len(article_times_valid)} valid articles across {len(timestamps)} timestamps")

    # Main vectorized outer loop
    for i, timestamp in enumerate(tqdm(timestamps, desc="Processing timestamps", leave=True)):
        cutoff_time = timestamp - np.timedelta64(int(lookback_window), 'h')
        time_mask = (article_times_valid >= cutoff_time) & (article_times_valid <= timestamp)
        if not time_mask.any():
            continue
        # Slice relevant articles
        valid_article_times = article_times_valid[time_mask]
        valid_article_topics = article_topics_valid[time_mask]
        # Get hours since publication
        hours_since = (timestamp - valid_article_times).astype('timedelta64[h]').astype(float)
        weights = np.exp(-decay_lambda * hours_since)
        # Accumulate by topic
        for topic in topics:
            topic_mask = valid_article_topics == topic
            weighted_counts_array[i, topic_to_idx[topic]] = np.sum(weights[topic_mask])

    for idx, topic in enumerate(topics):
        td_topics_df[topic] = weighted_counts_array[:, idx]

    return td_topics_df

# Compute time-decayed topic counts
td_topics_df_test = compute_time_decayed_topic_counts(news_df, master_df, lookback_window=DEFAULT_LOOKBACK_WINDOW, decay_lambda=DEFAULT_DECAY_LAMBDA)

### Sanity-Check Topic Aggregates

Checking the size and sample rows of the time-decayed topic matrix confirms the aggregation produced hourly features without gaps.

**Validation Steps:**
1. **Shape Consistency:**
   ```python
   assert topic_matrix.shape[0] == len(master_df)  # One row per hour
   assert topic_matrix.shape[1] == len(topic_labels)  # One column per topic
   ```

2. **No NaNs:** Aggregation should produce zeros (no articles) rather than missing values
   ```python
   assert topic_matrix.isna().sum().sum() == 0
   ```

3. **Value Ranges:** Counts should be non-negative and decay over time
   ```python
   assert (topic_matrix >= 0).all().all()
   print(topic_matrix.describe())  # Check mean/max are reasonable
   ```

4. **Temporal Monotonicity (Loose):** Recent hours should generally have higher counts
   ```python
   recent_mean = topic_matrix.iloc[-24:].mean().mean()  # Last 24 hours
   older_mean = topic_matrix.iloc[-168:-144].mean().mean()  # 7 days ago
   assert recent_mean >= older_mean * 0.5  # Allow some variance
   ```

**Example Output:**
```
           supply_shock_decayed  demand_decayed  weather_decayed  ...
2020-01-15       3.42              1.18            0.85          ...
2020-01-16       4.01              0.92            1.33          ...
```

**Red Flags:**
- **All Zeros:** No articles passed filters; check date range alignment
- **Uniform Values:** Decay not applied; λ may be set to 1.0 (no decay)
- **Spikes to Infinity:** Overflow from excessive article counts or λ>1

If checks pass, these features are ready to merge into `master_df`.

In [ ]:
print(td_topics_df_test.shape)
td_topics_df_test.head(5)

## Time-Decayed Embedding Aggregation

We apply the same decay logic to dense embeddings and optionally compress them with UMAP. The goal is to summarise the evolving semantic space in a fixed-size feature vector that downstream models can consume without incurring large dimensionality or recomputation costs.

**Process:**
1. **Decay Weighting (Same as Topics):**
   For each hour, compute weighted average of all article embeddings:
   ```python
   for timestamp in hourly_index:
       relevant_articles = news[news.published <= timestamp]
       hours_ago = (timestamp - relevant_articles.published).dt.total_seconds() / 3600
       weights = lambda_value ** hours_ago

       # Weighted average of 768-dimensional embeddings
       avg_embedding = (relevant_articles['embedding'] * weights[:, None]).sum(axis=0) / weights.sum()
       features[timestamp]['embedding_decayed'] = avg_embedding
   ```

2. **Dimensionality Reduction with UMAP:**
   768 dimensions can overwhelm models and slow training. UMAP (Uniform Manifold Approximation and Projection) compresses to, e.g., 10-50 dimensions while preserving semantic neighborhoods:
   - **Input:** `[num_hours, 768]` matrix of decayed embeddings
   - **Output:** `[num_hours, n_components]` reduced matrix
   - **Parameters:** `n_neighbors` (local vs global structure), `min_dist` (cluster tightness)

**Why UMAP Over PCA?**
- **Nonlinear:** Captures complex manifolds (PCA only handles linear relationships)
- **Preserves Clusters:** Articles about "nuclear outages" stay grouped even after reduction
- **Faster Than t-SNE:** Scales to 100K+ samples

**Trade-Offs:**
- **Information Loss:** Compressing 768→10 dimensions discards detail (but may remove noise)
- **Computation Cost:** UMAP fitting takes 1-5 minutes for 10K samples
- **Hyperparameter Sensitivity:** Poor `n_neighbors` choice can distort structure

**Grid Search:**
We'll test multiple reduction settings (e.g., 10, 20, 50 components) alongside decay rates to find the sweet spot between model performance and training speed.

**Output:**
Either the full 768-dimensional decayed embeddings (if skipping reduction) or compressed versions (e.g., `embedding_umap_10d`).

In [ ]:
def compute_time_decayed_embeddings(news_df, master_df, lookback_window=DEFAULT_LOOKBACK_WINDOW, decay_lambda=DEFAULT_DECAY_LAMBDA, verbose=True):
    """
    Compute time-decayed weighted average embeddings (OPTIMIZED VERSION).

    For each timestamp in master_df, compute the weighted average embedding of articles
    published within the lookback_window, using exponential time decay.
    Assumes news_df uses 'publishedAt' as its datetime index.

    Weight formula: weight = e^(-lambda * hours_since_publication)

    Parameters:
        news_df: DataFrame with 'publishedAt' as DatetimeIndex and an 'embedding' column
        master_df: DataFrame with datetime index representing target timestamps
        lookback_window: Number of hours to look back (default: 336h = 2 weeks)
        decay_lambda: Decay rate parameter (default: 0.05)
        verbose: Whether to print progress messages (default: True)

    Returns:
        np.ndarray: Array of weighted average embeddings with shape (n_timestamps, embedding_dim)
    """

    # Check that news_df index is datetime and named 'publishedAt'
    if not isinstance(news_df.index, pd.DatetimeIndex):
        raise ValueError("news_df must have a DatetimeIndex")
    if news_df.index.name != 'publishedAt':
        raise ValueError("news_df index must be named 'publishedAt'")

    td_embedding_news_df = news_df.copy()
    # Remove timezone if exists
    if td_embedding_news_df.index.tz is not None:
        td_embedding_news_df.index = td_embedding_news_df.index.tz_localize(None)

    if verbose:
        print(f"Computing time-decayed weighted average embeddings for {len(master_df)} timestamps")
        print(f"Lookback window: {lookback_window}h, decay lambda: {decay_lambda}")
        print("Using optimized vectorized computation...")

    # OPTIMIZATION 1: Vectorized preprocessing of embeddings
    valid_embeddings_mask = td_embedding_news_df['embedding'].notna()
    if not valid_embeddings_mask.any():
        raise ValueError("No valid embeddings found. Check that news_df has valid embeddings.")

    embedding_series = td_embedding_news_df.loc[valid_embeddings_mask, 'embedding']
    index_series = td_embedding_news_df.index[valid_embeddings_mask]

    article_embeddings_list: list[np.ndarray] = []
    for emb in embedding_series:
        if isinstance(emb, pd.Series):
            emb = emb.to_numpy()
        emb = np.asarray(emb, dtype=np.float32)
        if emb.ndim > 1:
            emb = emb.reshape(-1)
        article_embeddings_list.append(emb)

    if not article_embeddings_list:
        raise ValueError("All embeddings were empty after filtering valid rows.")

    embedding_lengths = {arr.shape[0] for arr in article_embeddings_list}
    if len(embedding_lengths) != 1:
        raise ValueError(f"Inconsistent embedding dimensions detected: {embedding_lengths}")

    embedding_dim = embedding_lengths.pop()
    article_embeddings_array = np.vstack(article_embeddings_list)

    article_times_valid = index_series.values.astype('datetime64[ns]')

    # Sort articles by time for efficient searching
    sort_indices = np.argsort(article_times_valid)
    article_times_valid = article_times_valid[sort_indices]
    article_embeddings_array = article_embeddings_array[sort_indices]

    # Initialize output array: shape (n_timestamps, embedding_dim)
    timestamps = master_df.index.values.astype('datetime64[ns]')
    weighted_embeddings_array = np.zeros((len(timestamps), embedding_dim), dtype=np.float32)

    if verbose:
        print(f"Processing {len(article_embeddings_array)} valid embeddings across {len(timestamps)} timestamps")
        print("Using binary search for efficient article lookup...")

    # OPTIMIZATION 2: Use binary search (searchsorted) to find relevant articles efficiently
    # Convert lookback_window to timedelta64
    lookback_delta = np.timedelta64(int(lookback_window), 'h')

    # Pre-compute cutoff times for all timestamps
    cutoff_times = timestamps - lookback_delta

    # Process timestamps with progress bar
    for i in tqdm(range(len(timestamps)), desc="Processing timestamps", disable=not verbose):
        timestamp = timestamps[i]
        cutoff_time = cutoff_times[i]

        # OPTIMIZATION 3: Use binary search to find articles in range [cutoff_time, timestamp]
        # searchsorted finds insertion points - very fast for sorted arrays
        start_idx = np.searchsorted(article_times_valid, cutoff_time, side='left')
        end_idx = np.searchsorted(article_times_valid, timestamp, side='right')

        if start_idx >= end_idx:
            # No articles in this window
            continue

        # Get articles in the time window
        window_embeddings = article_embeddings_array[start_idx:end_idx]
        window_times = article_times_valid[start_idx:end_idx]

        # OPTIMIZATION 4: Vectorized hours and weights calculation
        hours_since = (timestamp - window_times).astype('timedelta64[h]').astype(np.float32)
        weights = np.exp(-decay_lambda * hours_since, dtype=np.float32)

        # OPTIMIZATION 5: Efficient weighted sum using einsum or broadcasting
        total_weight = np.sum(weights)
        if total_weight > 1e-10:  # Avoid division by very small numbers
            # weights: (n_articles,), window_embeddings: (n_articles, embedding_dim)
            # Result: (embedding_dim,)
            weighted_sum = np.sum(weights[:, np.newaxis] * window_embeddings, axis=0)
            weighted_embeddings_array[i] = weighted_sum / total_weight
        else:
            weighted_embeddings_array[i] = np.zeros(embedding_dim, dtype=np.float32)

    if verbose:
        print(f"\nCompleted time-decayed aggregation")
        print(f"Embedding shape: {weighted_embeddings_array.shape}")

    return weighted_embeddings_array

# Compute time-decayed weighted average embeddings
weighted_embeddings_array_test = compute_time_decayed_embeddings(
    news_df, master_df, lookback_window=DEFAULT_LOOKBACK_WINDOW, decay_lambda=DEFAULT_DECAY_LAMBDA
)

### Embedding Reduction Helper

Reducing high-dimensional embeddings with caching keeps downstream models tractable. We encapsulate the logic so later cells just request a labelled reduction.

**Function Purpose:**
Wraps UMAP fitting and transformation with:
1. **Caching:** Saves fitted UMAP model and reduced embeddings to disk (parquet)
2. **Deterministic Seeding:** Ensures reproducible reductions across runs
3. **Error Handling:** Catches edge cases like insufficient samples (`n_samples < n_neighbors`)

**Typical Usage:**
```python
reduced_embeddings = reduce_embeddings(
    embeddings_768d,
    n_components=20,
    n_neighbors=15,
    cache_key='lambda_0.95_horizon_72h'
)
```

**Parameters:**
- `n_components`: Target dimensionality (10-50 typical)
- `n_neighbors`: Local neighborhood size (5-50; higher = more global structure)
- `min_dist`: Minimum distance between points in low-dim space (0.0-1.0; lower = tighter clusters)
- `metric`: Distance function ('cosine' for embeddings, 'euclidean' for other data)

**Cache Benefits:**
UMAP fitting is expensive (minutes for large datasets). Caching means:
- **Instant Reloads:** Re-running cells after a crash doesn't recompute
- **Consistent Comparisons:** Same reduction across parameter sweep iterations
- **Disk vs Memory Trade-Off:** 10-component reduced matrix is 76x smaller than original (768→10)

**When to Skip Reduction:**
If models train fast enough with full embeddings, skip reduction to avoid information loss. Test both paths (full vs reduced) in grid search.

In [ ]:
def reduce_embeddings_gpu_first(
    embeddings: np.ndarray,
    index: pd.Index,
    cache_label: str,
    n_components: int = 20,
    use_umap: bool = True,
) -> pd.DataFrame:
    """Reduce embeddings with GPU-first UMAP and disk caching."""
    import hashlib
    import json
    import pyarrow as pa
    import pyarrow.parquet as pq

    cache_root = Path('.cache/embeddings')
    if 'pipeline_config' in globals():
        try:
            cache_root = Path(globals()['pipeline_config'].artifact_dir) / 'embeddings'
        except AttributeError:
            pass
    cache_root.mkdir(parents=True, exist_ok=True)

    checksum = hashlib.sha1(embeddings.tobytes()).hexdigest()
    cache_path = cache_root / f'{cache_label}_reduction.parquet'
    manifest_path = cache_root / f'{cache_label}_reduction.json'

    if cache_path.exists() and manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
            if manifest.get('checksum') == checksum and manifest.get('components') == n_components:
                print(f"Loaded reduced embeddings from cache ({cache_path})")
                return pq.read_table(cache_path).to_pandas()
        except Exception as cache_exc:
            print(f"Cache read failed ({cache_exc}); recomputing")
            cache_path.unlink(missing_ok=True)
            manifest_path.unlink(missing_ok=True)

    backend = "pca"
    reduced = None

    if use_umap:
        if HAS_CUML_UMAP:
            device_info = detect_compute_device(task='embeddings', verbose=False)
            if device_info.get('device') == 'cuda':
                try:
                    import cupy as cp  # type: ignore

                    reducer = CUML_UMAP(
                        n_components=n_components,
                        n_neighbors=15,
                        min_dist=0.1,
                        random_state=RANDOM_STATE,
                    )
                    reduced_gpu = reducer.fit_transform(cp.asarray(embeddings))
                    reduced = cp.asnumpy(reduced_gpu)
                    backend = "cuml-umap"
                except Exception as gpu_exc:
                    print(f"cuML UMAP unavailable ({gpu_exc}); reverting to CPU UMAP.")
        if reduced is None:
            reducer = umap.UMAP(n_components=n_components, n_jobs=-1, verbose=False)
            reduced = reducer.fit_transform(embeddings)
            backend = "umap-learn"
    else:
        reducer = PCA(n_components=n_components)
        reduced = reducer.fit_transform(embeddings)
        backend = "pca"

    reduced_df = pd.DataFrame(
        reduced,
        index=index,
        columns=[f'embedding_dim_{i}' for i in range(reduced.shape[1])]
    )

    try:
        pq.write_table(pa.Table.from_pandas(reduced_df), cache_path)
        manifest_path.write_text(json.dumps({'checksum': checksum, 'components': n_components, 'backend': backend}))
    except Exception as write_exc:
        print(f"Reduction cache write skipped: {write_exc}")

    print(f"Embedding reduction backend: {backend}")
    return reduced_df


### Apply Dimensionality Reduction

We log the reduction workflow, handle NaNs defensively, and persist reduced embeddings so later runs reuse the computation.

**Step-by-Step:**
1. **Check for NaNs:**
   ```python
   if embeddings.isna().any():
       # Impute with column mean or drop rows
       embeddings = embeddings.fillna(embeddings.mean())
   ```

2. **Fit UMAP:**
   ```python
   reducer = umap.UMAP(n_components=n_components, random_state=SEED)
   reduced = reducer.fit_transform(embeddings)
   ```

3. **Validate Output:**
   ```python
   assert reduced.shape == (len(embeddings), n_components)
   assert not np.isnan(reduced).any()
   ```

4. **Cache to Disk:**
   ```python
   pd.DataFrame(reduced).to_parquet(cache_path)
   ```

**Logging:**
The profiling context captures:
- **Fit Time:** How long UMAP took (for cost-benefit analysis)
- **Memory Peak:** UMAP can spike to 5-10x input size during graph construction
- **Cache Path:** Where reduced embeddings were saved

**Defensive NaN Handling:**
Embeddings should never be NaN if generation succeeded, but defensively imputing prevents downstream crashes. Logging the imputation count helps detect upstream bugs.

**Output Validation:**
Before merging into `master_df`, confirm:
- Reduced embeddings align with hourly index (no off-by-one errors)
- Values are finite (UMAP can produce Inf with extreme parameter choices)
- Variance isn't zero (indicates collapsed dimensions; retry with different `n_neighbors`)

In [ ]:
# Apply GPU-aware dimensionality reduction to the time-decayed embeddings
print(f"Applying UMAP to reduce embeddings from {weighted_embeddings_array_test.shape[1]} to 20 dimensions...")

nan_mask = np.isnan(weighted_embeddings_array_test).any(axis=1)
num_nan_rows = nan_mask.sum()
print(f"Found {num_nan_rows} rows with NaN values. Filling with zeros before UMAP...")

weighted_embeddings_clean_test = weighted_embeddings_array_test.copy()
weighted_embeddings_clean_test[nan_mask] = 0.0

td_embeddings_df_test = reduce_embeddings_gpu_first(
    embeddings=weighted_embeddings_clean_test,
    index=master_df.index,
    cache_label='td_embeddings',
    n_components=20,
    use_umap=True,
)

print(f"Output shape: {td_embeddings_df_test.shape}")


### Inspect Reduced Embeddings

We confirm the reduced matrix dimensions before merging to catch any silent mismatches between embeddings and master index.

**Critical Checks:**
1. **Row Count Match:**
   ```python
   assert len(reduced_embeddings) == len(master_df)
   # Every hour must have a reduced embedding
   ```

2. **Index Alignment:**
   ```python
   assert (reduced_embeddings.index == master_df.index).all()
   # Timestamps must match exactly (not just count)
   ```

3. **Column Names:**
   ```python
   expected_cols = [f'embedding_umap_{i}' for i in range(n_components)]
   assert list(reduced_embeddings.columns) == expected_cols
   ```

4. **Value Sanity:**
   ```python
   print(reduced_embeddings.describe())
   # Mean: ~0 (UMAP output is centered)
   # Std: ~1-5 (typical range)
   # Min/Max: Check for extreme outliers (|z| > 10)
   ```

**Why Alignment Matters:**
If embedding row 100 corresponds to hour 2020-01-15 12:00 but `master_df` row 100 is 2020-01-15 13:00, the model trains on misaligned data—learning spurious correlations.

**Common Mismatches:**
- **Dropped Rows:** Energy data had NaNs that were dropped, but embeddings weren't
- **Timezone Differences:** Embeddings indexed in UTC, energy data in Berlin time
- **Duplicate Timestamps:** Multiple articles per hour collapsed incorrectly

**Fix Strategy:**
```python
# Safe merge on index
master_df = master_df.join(reduced_embeddings, how='inner')
# Inner join ensures only matching timestamps proceed
```

If this check passes, proceed to feature assembly.

In [ ]:
print(td_embeddings_df_test.shape)
td_embeddings_df_test.head(5)

## Feature Assembly

With all engineered signals ready, we merge them into the master frame. This step locks in a single table per timestamp so later model training can focus on sampling strategies rather than ad-hoc joins.

**Merge Process:**
```python
master_df = (energy_features
             .join(topic_aggregates, how='left')
             .join(reduced_embeddings, how='left')
             .join(baseline_lags, how='left'))
```

**Join Strategy (Left vs Inner vs Outer):**
- **Left:** Keeps all energy timestamps, fills NaN where news is missing (reasonable for sparse news days)
- **Inner:** Only keeps timestamps with ALL features present (safest, loses some data)
- **Outer:** Keeps everything, maximal NaNs (dangerous for models)

We typically use **left join** with `fillna(0)` for news features, interpreting "no articles" as "zero topic count" rather than missing data.

**Final Feature Set:**
After assembly, `master_df` contains:
1. **Time Features:** hour, day_of_week, month (12-15 columns)
2. **Price Lags:** 1h, 3h, 6h, 24h, 72h lags and moving averages (~20 columns)
3. **Supply/Demand:** Wind, solar, load, imports (~10 columns)
4. **Topic Aggregates:** Decayed counts per topic (~6 columns)
5. **Reduced Embeddings:** UMAP components (~10-50 columns)
6. **Target:** `spread_direction` (Long/Neutral/Short)

**Total:** ~60-100 features depending on configuration.

**Post-Merge Validation:**
```python
# Check for unexpected NaNs
print(master_df.isna().sum())

# Confirm target is present
assert 'spread_direction' in master_df.columns

# Drop rows with NaN target (undefined labels)
master_df = master_df.dropna(subset=['spread_direction'])
```

**Memory Footprint:**
With 100K hourly timestamps and 80 features:
- Float64: 100K × 80 × 8 bytes = 64 MB (manageable)
- Includes embeddings: add ~100K × 768 × 8 = 614 MB if not reduced

If memory becomes an issue, switch to float32 or drop unnecessary columns (e.g., intermediate calculation artifacts).

In [ ]:
# Store the merged_df_test for baseline reference
merged_df_test = master_df.join([td_topics_df_test, td_embeddings_df_test], how='left')
print(merged_df_test.shape)
merged_df_test.head(5)

### Guard Disk Usage

Before materialising additional combinations we re-check free disk space to avoid cache corruption.

**Why Monitor Disk Space?**
The upcoming grid search will generate dozens of feature matrices (each 50-200 MB) plus UMAP models, telemetry logs, and model checkpoints. Running out of disk mid-process causes:
- **Corrupted Cache Files:** Partial writes that look valid but contain garbage
- **Lost Computation:** Hours of feature generation discarded
- **Silent Failures:** Python may not raise an error until you try to read the corrupted file

**Minimum Threshold:**
We check for at least 5-10 GB free before proceeding. This buffer accounts for:
- Temporary files (UMAP creates intermediate graphs)
- Log growth (telemetry can accumulate megabytes per stage)
- OS overhead (filesystem metadata, swap space)

**Typical Disk Usage Breakdown:**
- Cached embeddings: 300-600 MB
- Precomputed feature matrices: 50-200 MB each × 20 grid points = 1-4 GB
- UMAP models: 10-50 MB each
- XGBoost checkpoints: 50-200 MB
- Logs and telemetry: 100-500 MB

**Failure Action:**
If insufficient space:
1. Clean up old experiment artifacts
2. Move embeddings to external storage and symlink
3. Reduce grid search size (fewer λ values or horizons)
4. Use streaming pipeline without caching

**Code Pattern:**
```python
import shutil
free_gb = shutil.disk_usage('.').free / (1024**3)
assert free_gb > 5, f"Only {free_gb:.1f} GB free, need 5+ GB"
```

In [ ]:
ensure_disk_headroom(Path('.'), min_free_gb=5.0)


---
# Grid Search for Time-Decayed Parameters

We next sweep key decay parameters to understand which temporal fingerprints carry predictive power. By precomputing combinations up front we constrain later model fitting to a manageable set of candidates without constantly recomputing heavy aggregations.

In [ ]:
# TODO: In future runs add more parameters to the grid search

# Define parameter grid for grid search
lookback_windows = [24, 72, 168, 336]  # 1, 2 weeks in hours
decay_lambdas = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]

# Create all combinations
param_combinations = []
for lw in lookback_windows:
    for dl in decay_lambdas:
        param_combinations.append({'lookback_window': lw, 'decay_lambda': dl})

print(f"Total parameter combinations: {len(param_combinations)}")
print(f"Lookback windows: {lookback_windows}")
print(f"Decay lambdas: {decay_lambdas}")
print(f"\nFirst 3 combinations (showing sample):")
for i, combo in enumerate(param_combinations[:3], 1):
    print(f"{i}. lookback_window={combo['lookback_window']}h, decay_lambda={combo['decay_lambda']}")


### Precomputation Wrapper

This helper precomputes topic and embedding features for a single hyperparameter pair so we can parallelise the grid search cleanly.

**Function Signature:**
```python
def precompute_features(lambda_decay, time_horizon_hours):
    # Generate time-decayed topic counts with given λ and horizon
    topic_features = aggregate_topics(lambda_decay, time_horizon_hours)

    # Generate time-decayed embeddings (optionally reduce with UMAP)
    embedding_features = aggregate_embeddings(lambda_decay, time_horizon_hours)

    # Merge and save to cache
    combined = topic_features.join(embedding_features)
    cache_path = f'cache/features_lambda_{lambda_decay}_horizon_{time_horizon_hours}.parquet'
    combined.to_parquet(cache_path)

    return cache_path
```

**Why Wrap This?**
- **Parallelization:** Each parameter pair is independent; we can run 8 combinations on 8 CPU cores
- **Error Isolation:** If one parameter set fails (e.g., λ=1.0 causes overflow), others proceed
- **Progress Tracking:** Log completion of each wrapper call to estimate remaining time

**Inputs:**
- `lambda_decay`: Exponential decay rate (0.85-0.98)
- `time_horizon_hours`: How far back to look (12, 24, 48, 72 hours)

**Outputs:**
- Parquet file path containing the merged features
- Side effect: Profiling log entry for this parameter set

**Grid Search Example:**
```python
param_grid = [
    (0.90, 24), (0.90, 48), (0.90, 72),
    (0.95, 24), (0.95, 48), (0.95, 72),
]
# Each tuple will be passed to precompute_features in parallel
```

This wrapper is called by the next cell's parallel executor.

In [ ]:
def precompute_single_combination(params):
    """Helper function to precompute features for a single parameter combination."""
    lw = params['lookback_window']
    dl = params['decay_lambda']
    
    # Compute topic counts
    td_topics = compute_time_decayed_topic_counts(news_df, master_df, lookback_window=lw, decay_lambda=dl, verbose=False)
    
    # Compute embeddings
    weighted_embeddings = compute_time_decayed_embeddings(news_df, master_df, lookback_window=lw, decay_lambda=dl, verbose=False)
    
    # Handle NaN values before dimensionality reduction
    nan_mask = np.isnan(weighted_embeddings).any(axis=1)
    weighted_embeddings_clean = weighted_embeddings.copy()
    weighted_embeddings_clean[nan_mask] = 0.0
    
    td_embeddings = reduce_embeddings_gpu_first(
        embeddings=weighted_embeddings_clean,
        index=master_df.index,
        cache_label=f'td_embeddings_lw{lw}_dl{dl}',
        n_components=20,
        use_umap=True,
    )
    
    return (lw, dl), (td_topics, td_embeddings)

### Parallel Feature Materialisation

We fan out the heavy precomputation work across CPU cores to shave time off the upcoming grid search, accepting the upfront cost once.

**Why Parallelize?**
Sequential computation for 20 parameter combinations:
- Single-threaded: 5 min/combination × 20 = 100 minutes
- 8 cores parallel: 5 min/combination × 20/8 = 12.5 minutes (8x speedup)

**Python Multiprocessing:**
```python
from multiprocessing import Pool

with Pool(processes=8) as pool:
    cache_paths = pool.starmap(precompute_features, param_grid)
```

**Considerations:**
- **Memory per Worker:** Each process loads the full news dataset; ensure 8 × RAM_needed < total_RAM
- **GIL (Global Interpreter Lock):** Not an issue here since we're doing computation-heavy work (NumPy/Pandas release GIL)
- **Shared Disk I/O:** All workers write to disk; SSD recommended to avoid bottleneck

**Progress Monitoring:**
Since workers run in separate processes, use:
- **Callback Functions:** Print when each parameter set completes
- **Profiling Logs:** Each worker writes to a separate log file
- **`tqdm` with `imap`:** Show progress bar across all workers

**Alternative: Dask or Ray:**
For distributed computation across multiple machines:
```python
import dask
futures = [dask.delayed(precompute_features)(λ, h) for λ, h in param_grid]
results = dask.compute(*futures)
```

**Output:**
List of cache file paths, one per parameter combination, ready for Ridge screening.

In [ ]:
# Precompute features for all parameter combinations
# This will compute time-decayed topics and embeddings for each combination
print(f"Precomputing features for {len(param_combinations)} parameter combinations...")
print("This may take a while, but will speed up the grid search significantly...\n")

# Use parallel processing to precompute features
precomputed_features = dict(
    Parallel(n_jobs=-1, verbose=10)(
        delayed(precompute_single_combination)(params)
        for params in param_combinations
    )
)

print(f"\nPrecomputation complete! Features computed for {len(precomputed_features)} parameter combinations.")
print(f"Sample keys: {list(precomputed_features.keys())[:3]}")

### Assemble Train/Val/Test Datasets

Within a profiled stage we merge features, drop invalid rows, and persist deterministic splits for every parameter set. This isolates data wrangling from model evaluation.

**Why Separate Assembly from Training?**
- **Reproducibility:** Same splits across all models (fair comparison)
- **Efficiency:** Splitting once is faster than re-splitting in each CV fold
- **Debugging:** Inspect split files manually to confirm data quality

**Process:**
1. **Load Cached Features:**
   ```python
   features = pd.read_parquet(cache_path)
   master = energy_baseline.join(features, how='left')
   ```

2. **Drop Invalid Rows:**
   ```python
   # Remove rows with NaN target or excessive feature NaNs
   master = master.dropna(subset=['spread_direction'])
   master = master[master.isna().sum(axis=1) < max_nans_per_row]
   ```

3. **Temporal Split (Crucial for Time-Series):**
   ```python
   # Sort by timestamp (should already be sorted, but verify)
   master = master.sort_index()

   # Split by date, not random sampling
   train_end = '2021-12-31'
   val_end = '2022-06-30'

   train = master[:train_end]
   val = master[train_end:val_end]
   test = master[val_end:]
   ```

4. **Persist Splits:**
   ```python
   train.to_parquet(f'datasets/train_{param_id}.parquet')
   val.to_parquet(f'datasets/val_{param_id}.parquet')
   test.to_parquet(f'datasets/test_{param_id}.parquet')
   ```

**Validation:**
```python
# Check no overlap
assert train.index.max() < val.index.min()
assert val.index.max() < test.index.min()

# Check class balance
print(train['spread_direction'].value_counts(normalize=True))
```

**Output:**
Directory structure:
```
datasets/
  train_lambda_0.90_horizon_24.parquet
  val_lambda_0.90_horizon_24.parquet
  test_lambda_0.90_horizon_24.parquet
  ...
```
Each file is a self-contained dataset ready for model training.

In [ ]:
with StageProfiler("Stage 3 — Dataset assembly", device_config):
    # Merge and split all precomputed datasets
    # This eliminates redundant merging and splitting operations later
    print("Merging and splitting all precomputed datasets...")
    print("This will create 70% train / 20% validation / 10% test splits for each dataset\n")

    preprocessed_datasets = {}
    master_df_pre_name_map = {}

    for idx, (params_key, (td_topics_df, td_embeddings_df)) in enumerate(precomputed_features.items(), start=1):
        # Step 1: Merge topics and embeddings with master_df ONCE
        merged_features_df = master_df.join([td_topics_df, td_embeddings_df], how='left')

        # Step 2: Drop rows with NaN targets ONCE (classification target)
        model_df = merged_features_df.dropna(subset=['spread_target_shift_24']).copy()
        dataset_name = f"master_df_pre_{idx}"

        # Persist the full dataframe for this parameter combination
        globals()[dataset_name] = model_df
        master_df_pre_name_map[params_key] = dataset_name

        # Step 3: Split into train (70%), validation (20%), test (10%) ONCE
        train_size = int(len(model_df) * 0.7)
        val_size = int(len(model_df) * 0.2)
        train_df = model_df.iloc[:train_size].copy()
        val_df = model_df.iloc[train_size:train_size + val_size].copy()
        test_df = model_df.iloc[train_size + val_size:].copy()

        # Store split dataframes as standalone variables for convenience
        globals()[f"{dataset_name}_train"] = train_df
        globals()[f"{dataset_name}_val"] = val_df
        globals()[f"{dataset_name}_test"] = test_df

        # Get feature columns for later use
        topic_cols = list(td_topics_df.columns)
        embedding_cols = list(td_embeddings_df.columns)
        news_features = topic_cols + embedding_cols

        # Store preprocessed data
        preprocessed_datasets[params_key] = {
            'dataset_name': dataset_name,
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'news_features': news_features,
            'topic_cols': topic_cols,
            'embedding_cols': embedding_cols
        }

    print(f"Preprocessing complete! Processed {len(preprocessed_datasets)} datasets.")

    if preprocessed_datasets:
        sample_key = next(iter(preprocessed_datasets))
        sample_name = master_df_pre_name_map[sample_key]
        sample_data = preprocessed_datasets[sample_key]
        print(f"Sample dataset: {sample_name}")
        print(f"  Train: {len(sample_data['train_df'])} samples")
        print(f"  Validation: {len(sample_data['val_df'])} samples")
        print(f"  Test: {len(sample_data['test_df'])} samples")


## 📊 Exploratory Data Analysis

Before tuning models we visualise the merged dataset to confirm the engineered signals align with domain expectations (price spreads, class balance, news volume, correlations). Exploratory Data Analysis (EDA) is the practice of using statistical graphics and summary statistics to understand data before applying machine learning.

**Why EDA Before Modeling?**
Jumping straight into model training without EDA is risky:
- **Garbage In, Garbage Out:** If features are constant, duplicated, or nonsensical, no amount of hyperparameter tuning will help
- **Data Leakage Detection:** Suspiciously high correlations (>0.99) between features and target suggest future information leaked into training
- **Sanity Checks:** Does renewable generation spike during summer (solar)? Do news topic counts surge during crises?

**Our EDA Roadmap:**
The next cells will produce diagnostic visualizations and statistics:

1. **Target Distribution:**
   - Class balance across Long/Neutral/Short
   - Check for temporal drift (e.g., 2020 has different balance than 2022)

2. **Feature Distributions:**
   - Histograms for price lags, news counts, embedding dimensions
   - Identify outliers (e.g., 99th percentile price 10x higher than median)

3. **Correlation Analysis:**
   - Heatmap of feature correlations
   - Flag redundant pairs (e.g., `price_lag_1h` and `price_lag_2h` at r=0.98)

4. **Time-Series Plots:**
   - Price spreads over time (check for regime changes)
   - News volume over time (check for scraper outages)

5. **Feature-Target Relationships:**
   - Box plots: How do Long/Neutral/Short classes differ in `renewable_share`?
   - Scatter plots: Does higher `supply_shock_decayed` correlate with larger spreads?

**Actionable Outcomes:**
- **Drop Features:** Remove constants or near-duplicates (reduces model complexity)
- **Handle Outliers:** Winsorize (cap) extreme values or investigate as data errors
- **Adjust Target:** If class balance is 10%/80%/10%, revisit neutral band threshold
- **Confidence Boost:** If EDA reveals sensible patterns (e.g., "Long" class has higher recent volatility), we trust the feature engineering worked

**Note:** The actual EDA code cell follows this markdown. We place this explanation here to clarify *why* we're pausing model development to visualize data.

In [ ]:
# ============================================================================
# EXPLORATORY DATA ANALYSIS (EDA) VISUALIZATIONS
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Helper function to get correct energy price columns
def get_column_name(possible_names, df):
    """Return the first matching name in possible_names that exists in df.columns."""
    for name in possible_names:
        if name in df.columns:
            return name
    return None

# Try to find correct columns for the auction and spot prices, robust to variant casing/format
day_ahead_col = get_column_name(['day_ahead_price', 'Day Ahead Auction', 'day ahead price', 'Day-ahead Price', 'day_ahead_auction'], master_df)
spot_price_col = get_column_name(['spot_price', 'Spot Price', 'spot price'], master_df)

if day_ahead_col is None or spot_price_col is None:
    raise KeyError(f"Could not find both required price columns in master_df. Found: {day_ahead_col} and {spot_price_col}")

# Determine time axis safely (column vs index)
if 'timestamp' in master_df.columns:
    time_axis = pd.to_datetime(master_df['timestamp'])
elif 'Timestamp' in master_df.columns:
    time_axis = pd.to_datetime(master_df['Timestamp'])
elif isinstance(master_df.index, pd.DatetimeIndex):
    time_axis = master_df.index
else:
    raise KeyError("No timestamp column or datetime index found in master_df.")

# Create comprehensive EDA dashboard
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Comprehensive Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.00)

# 1. Time series of prices
axes[0, 0].plot(time_axis, master_df[day_ahead_col],
                label='Day-ahead Price', alpha=0.7, linewidth=0.8)
axes[0, 0].plot(time_axis, master_df[spot_price_col],
                label='Spot Price', alpha=0.7, linewidth=0.8)
axes[0, 0].set_title('Price Time Series', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price (EUR/MWh)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Spread distribution
spread_series = master_df[day_ahead_col] - master_df[spot_price_col]
spread_data = spread_series.dropna()
axes[0, 1].hist(spread_data, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Spread Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Spread (Day-ahead - Spot, EUR/MWh)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero spread')
axes[0, 1].legend()

# 3. Target class balance
target_col = get_column_name(
    ['spread_target_shift_24', 'spread_target', 'target'], master_df)
if target_col is not None:
    target_counts = master_df[target_col].value_counts().sort_index()
    color_palette = ['#d62728', '#ff7f0e', '#2ca02c']
    if len(target_counts) > 3:
        color_palette = color_palette + ['#9467bd', '#8c564b']
    axes[1, 0].bar(target_counts.index, target_counts.values,
                   edgecolor='black', alpha=0.7, color=color_palette[:len(target_counts)])
    axes[1, 0].set_title('Target Class Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Target Class')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_xticks(target_counts.index)
    for i, v in enumerate(target_counts.values):
        axes[1, 0].text(target_counts.index[i], v, f'{v:,}\n({v/len(master_df)*100:.1f}%)',
                        ha='center', va='bottom', fontweight='bold')
else:
    axes[1, 0].text(0.5, 0.5, 'Target column not found',
                    ha='center', va='center', fontsize=12)
    axes[1, 0].set_title('Target Class Distribution', fontsize=12, fontweight='bold')

# 4. Articles per day (if news_df exists)
if 'news_df' in dir() and news_df is not None and len(news_df) > 0:
    # Assuming news_df has a datetime index or timestamp column
    if hasattr(news_df, 'index') and isinstance(news_df.index, pd.DatetimeIndex):
        articles_per_day = news_df.resample('D').size()
    else:
        # Try to find timestamp column
        time_col = [c for c in news_df.columns if 'time' in c.lower() or 'date' in c.lower()]
        if time_col:
            articles_per_day = news_df.set_index(pd.to_datetime(news_df[time_col[0]])).resample('D').size()
        else:
            articles_per_day = pd.Series([len(news_df)], index=[pd.Timestamp.now()])

    axes[1, 1].plot(articles_per_day.index, articles_per_day.values, linewidth=1.5)
    axes[1, 1].set_title('News Articles per Day', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('Article Count')
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'News data not available',
                    ha='center', va='center', fontsize=12)
    axes[1, 1].set_title('News Articles per Day', fontsize=12, fontweight='bold')

# 5. Correlation heatmap of energy features
energy_features = []
for price_col in [day_ahead_col, spot_price_col]:
    if price_col is not None:
        energy_features.append(price_col)
# Add renewable energy columns if they exist
for col in ['solar', 'wind_onshore', 'wind_offshore', 'biomass', 'hydro']:
    if col in master_df.columns:
        energy_features.append(col)

if len(energy_features) >= 2:
    corr_matrix = master_df[energy_features].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                ax=axes[2, 0], square=True, linewidths=1, cbar_kws={'shrink': 0.8})
    axes[2, 0].set_title('Energy Feature Correlation', fontsize=12, fontweight='bold')
else:
    axes[2, 0].text(0.5, 0.5, 'Insufficient features for correlation',
                    ha='center', va='center', fontsize=12)
    axes[2, 0].set_title('Energy Feature Correlation', fontsize=12, fontweight='bold')

# 6. Spread over time
axes[2, 1].plot(time_axis, spread_series, linewidth=0.8, alpha=0.7)
axes[2, 1].set_title('Spread Over Time', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Date')
axes[2, 1].set_ylabel('Spread (EUR/MWh)')
axes[2, 1].axhline(0, color='red', linestyle='--', alpha=0.5, linewidth=2)
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print data summary
print(f"\n{'='*60}")
print("DATA SUMMARY")
print(f"{'='*60}")
print(f"Date range: {time_axis.min()} to {time_axis.max()}")
print(f"Total hours: {len(master_df):,}")
if 'news_df' in dir() and news_df is not None:
    print(f"Total news articles: {len(news_df):,}")
if target_col is not None:
    print(f"\nTarget distribution:")
    for idx, count in master_df[target_col].value_counts().sort_index().items():
        # format idx safely: if it is int, print as int, else use general format
        if isinstance(idx, (int,)) or (isinstance(idx, float) and idx.is_integer()):
            idx_str = f"{int(idx):2d}"
        else:
            idx_str = f"{idx}"
        print(f"  Class {idx_str}: {count:6,} ({count/len(master_df)*100:5.1f}%)")
else:
    print("\nTarget column not found for distribution statistics.")
print(f"\nPrice statistics (EUR/MWh):")
print(f"  Day-ahead: mean={master_df[day_ahead_col].mean():.2f}, "
      f"std={master_df[day_ahead_col].std():.2f}")
print(f"  Spot:      mean={master_df[spot_price_col].mean():.2f}, "
      f"std={master_df[spot_price_col].std():.2f}")
print(f"  Spread:    mean={spread_data.mean():.2f}, std={spread_data.std():.2f}")
print(f"{'='*60}\n")

### Scale News Features

We standardise the news-derived features per dataset so downstream models compare like-for-like inputs regardless of parameter choice.

**What is Standardization?**
Transform features to zero mean and unit variance:
```python
X_scaled = (X - X.mean()) / X.std()
```

**Why Scale?**
1. **Model Convergence:** Gradient descent (in neural nets or logistic regression) converges faster when features are on similar scales
2. **Regularization Fairness:** Ridge/Lasso penalties treat all features equally only if they're on the same scale (else large-magnitude features get under-penalized)
3. **Interpretability:** Coefficients represent "effect of 1 standard deviation change" rather than "effect of 1 unit change in arbitrary units"

**Which Features to Scale?**
- **Yes:** Topic counts (range 0-50), embedding dimensions (range -5 to +5), price lags (range 20-100 EUR/MWh)
- **No:** Binary flags (is_weekend), categorical encodings (one-hot vectors), pre-normalized embeddings (if already length-1)

**Per-Dataset Scaling (Critical):**
```python
# Fit scaler on training set
scaler = StandardScaler()
scaler.fit(X_train[news_columns])

# Apply to train, val, test
X_train[news_columns] = scaler.transform(X_train[news_columns])
X_val[news_columns] = scaler.transform(X_val[news_columns])
X_test[news_columns] = scaler.transform(X_test[news_columns])
```

**Common Mistake (Data Leakage):**
❌ Fitting scaler on entire dataset before splitting:
```python
# BAD: Test set statistics influence training
X_all_scaled = StandardScaler().fit_transform(X_all)
```
This leaks information about the test set's mean/variance into training.

✅ Correct: Fit on train, transform train/val/test separately.

**Output:**
Scaled datasets where news features have mean ≈ 0, std ≈ 1. Baseline energy features (already on reasonable scales) remain untouched.

In [ ]:
with StageProfiler("Stage 3 — Dataset scaling", device_config):
    # Pre-scale all 20 preprocessed datasets
    # Fit scaler on training set and transform train/validation/test sets
    print("Pre-scaling all 20 datasets...")
    print("Fitting scaler on training set and transforming all splits\n")

    for params_key, data_dict in preprocessed_datasets.items():
        train_df = data_dict['train_df']
        val_df = data_dict['val_df']
        test_df = data_dict['test_df']
        news_features = data_dict['news_features']

        # Fit scaler on training set's news features only
        scaler_news = StandardScaler()
        train_news_features = train_df[news_features].fillna(0)
        scaler_news.fit(train_news_features)

        # Transform all splits using the fitted scaler
        # We'll create new columns with '_scaled' suffix for the news features
        for df_name, df in [('train_df', train_df), ('val_df', val_df), ('test_df', test_df)]:
            news_data = df[news_features].fillna(0)
            news_scaled = scaler_news.transform(news_data)

            # Create scaled column names
            for idx, feat in enumerate(news_features):
                scaled_col = f"{feat}_scaled"
                df[scaled_col] = news_scaled[:, idx]

            # Update the dataframe in the dictionary
            data_dict[df_name] = df

        # Store the scaler for later use
        data_dict['scaler_news'] = scaler_news

        # Store scaled news feature names
        scaled_news_features = [f"{feat}_scaled" for feat in news_features]
        data_dict['scaled_news_features'] = scaled_news_features

    print("Scaling complete! All datasets have been scaled.")
    # Get sample scaled feature count from first dataset
    sample_key = list(preprocessed_datasets.keys())[0]
    sample_scaled_features = preprocessed_datasets[sample_key]['scaled_news_features']
    print(f"Scaled news features created: {len(sample_scaled_features)} features per dataset")


### Ridge Evaluation Helper

This function evaluates a single parameter combination using time-aware CV. Documenting the guardrails here prevents silent failures when class balance is poor.

**What is Ridge Regression?**
Linear model with L2 regularization penalty:
```
Loss = Σ(y - Xβ)² + α·Σβ²
```
where α controls regularization strength. Higher α → smaller coefficients → less overfitting.

**Why Use Ridge for Screening?**
- **Speed:** Fits in seconds vs minutes for XGBoost (important when testing 20+ parameter sets)
- **Interpretability:** Coefficients reveal which features correlate with target (sanity check before heavy tuning)
- **Baseline:** If Ridge performs well, complex models may not be necessary

**Evaluation Process:**
```python
def evaluate_ridge(X_train, y_train, X_val, y_val):
    # Expanding window CV on training set
    cv_scores = []
    for train_idx, val_idx in expanding_window_split(X_train):
        ridge = RidgeClassifier(alpha=1.0)
        ridge.fit(X_train[train_idx], y_train[train_idx])
        score = ridge.score(X_train[val_idx], y_train[val_idx])
        cv_scores.append(score)

    # Final fit and validation score
    ridge.fit(X_train, y_train)
    val_score = ridge.score(X_val, y_val)

    return {
        'cv_mean': np.mean(cv_scores),
        'cv_std': np.std(cv_scores),
        'val_score': val_score
    }
```

**Guardrails:**
1. **Minimum Class Size:** Skip CV fold if any class has <5 samples (would cause singular matrices)
2. **Convergence Check:** Warn if Ridge solver doesn't converge (suggests poorly scaled features)
3. **Sanity Bounds:** Flag if accuracy <33% (worse than random guessing on 3-class problem)

**Output:**
DataFrame with columns: `[param_id, cv_mean, cv_std, val_score]`, sorted by `val_score` descending. Top 5-10 advance to XGBoost tuning.

In [ ]:
TARGET_COLUMN = 'spread_target_shift_24'
DEFAULT_ALPHAS = np.logspace(-3, 3, 13)


def evaluate_single_parameter_combination(params_key, data_dict, baseline_features,
                                          alphas=None, max_splits=5):
    """Evaluate a single parameter combination using expanding-window RidgeCV confined to the training split."""
    lw, dl = params_key
    dataset_name = data_dict['dataset_name']
    train_df = data_dict['train_df']
    val_df = data_dict['val_df']
    scaled_news_features = data_dict['scaled_news_features']

    feature_columns = baseline_features + scaled_news_features
    missing_features = [col for col in feature_columns if col not in train_df.columns]
    if missing_features:
        raise ValueError(f"Missing features {missing_features} in dataset {dataset_name}")

    if alphas is None:
        alphas = DEFAULT_ALPHAS

    X_train = train_df[feature_columns].fillna(0)
    y_train = train_df[TARGET_COLUMN].astype(int)
    X_val = val_df[feature_columns].fillna(0)
    y_val = val_df[TARGET_COLUMN].astype(int)

    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Training split lacks class diversity'
        }

    if len(X_val) == 0:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Validation split is empty'
        }

    max_possible_splits = max(0, len(X_train) - 1)
    effective_splits = min(max_splits, max_possible_splits)

    if effective_splits < 2:
        return {
            'lookback_window': lw,
            'decay_lambda': dl,
            'dataset_name': dataset_name,
            'best_alpha': None,
            'val_accuracy': np.nan,
            'val_macro_f1': np.nan,
            'model': None,
            'params_key': params_key,
            'skip_reason': 'Insufficient data for expanding-window CV'
        }

    tscv = TimeSeriesSplit(n_splits=effective_splits)

    ridge_cv = RidgeClassifierCV(alphas=alphas, cv=tscv, scoring='accuracy')
    ridge_cv.fit(X_train, y_train)

    val_predictions = ridge_cv.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_predictions)
    val_macro_f1 = f1_score(y_val, val_predictions, average='macro', zero_division=0)

    return {
        'lookback_window': lw,
        'decay_lambda': dl,
        'dataset_name': dataset_name,
        'best_alpha': ridge_cv.alpha_,
        'val_accuracy': val_accuracy,
        'val_macro_f1': val_macro_f1,
        'model': ridge_cv,
        'params_key': params_key,
        'skip_reason': None
    }


def grid_search_time_decay_params(preprocessed_datasets, baseline_features,
                                  alphas=None, max_splits=5):
    """Run expanding-window RidgeCV on each precomputed dataset and rank by validation performance."""
    print(f"Grid searching {len(preprocessed_datasets)} parameter combinations...")
    print(f"Using expanding-window RidgeCV confined to training splits (target: {TARGET_COLUMN})")
    print(f"Parallelizing evaluation across parameter combinations using joblib...\n")

    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluate_single_parameter_combination)(
            params_key,
            data_dict,
            baseline_features,
            alphas,
            max_splits
        )
        for params_key, data_dict in preprocessed_datasets.items()
    )

    # Filter out combinations that could not be evaluated
    valid_results = [res for res in results if res['skip_reason'] is None]
    invalid_results = [res for res in results if res['skip_reason'] is not None]

    if invalid_results:
        print("The following combinations were skipped:")
        for res in invalid_results:
            print(f"  - {res['dataset_name']} (lookback={res['lookback_window']}, lambda={res['decay_lambda']}): {res['skip_reason']}")
        print()

    if not valid_results:
        print("No valid results to rank. Please review the dataset or parameters.")
        return []

    # Sort by validation accuracy (descending), then macro F1
    results_sorted = sorted(
        valid_results,
        key=lambda x: (x['val_accuracy'], x['val_macro_f1']),
        reverse=True
    )

    globals()['ridgecv_valid_results'] = valid_results
    globals()['ridgecv_results_sorted'] = results_sorted

    top_results = results_sorted[:3]

    print(f"{'='*80}")
    print("TOP 3 PARAMETER COMBINATIONS:")
    print(f"{'='*80}")
    for idx, result in enumerate(top_results, 1):
        print(
            f"{idx}. dataset={result['dataset_name']} | lookback={result['lookback_window']}h | "
            f"lambda={result['decay_lambda']} | alpha={result['best_alpha']:.4f} | "
            f"Val Accuracy={result['val_accuracy']:.3f} | Val Macro-F1={result['val_macro_f1']:.3f}"
        )

    return top_results


### Baseline Feature Definition

We list the deterministic price/load lags that every model receives, keeping a clear separation between core telemetry features and news augmentations.

**Baseline Features (No News):**
1. **Price Lags:**
   - `price_lag_1h`, `price_lag_3h`, `price_lag_6h`, `price_lag_24h`
   - Captures short-term momentum and daily seasonality

2. **Rolling Price Statistics:**
   - `price_ma_24h`, `price_ma_7d` (moving averages smooth noise)
   - `price_std_24h` (volatility proxy)

3. **Load (Demand) Features:**
   - `load_lag_1h`, `load_lag_24h`
   - `load_ma_7d`

4. **Supply Features:**
   - `wind_generation`, `solar_generation`, `renewable_share`
   - `net_imports` (cross-border flows)

5. **Temporal Features:**
   - `hour_of_day` (0-23), `day_of_week` (0-6), `month` (1-12)
   - `is_weekend` (binary flag)

6. **Spread History:**
   - `spread_lag_1h`, `spread_lag_24h` (past forecast errors)

**Why Separate Baseline?**
- **Ablation Studies:** Compare "baseline only" vs "baseline + news" to quantify news value
- **Fallback Model:** If news data is unavailable (e.g., scraper fails), baseline model can still run
- **Feature Importance Analysis:** Identify if news features actually matter or if price lags dominate

**Typical Baseline Performance:**
- Ridge accuracy: 45-55% (better than random 33%, but not stellar)
- XGBoost accuracy: 55-65% (captures nonlinear price patterns)

If baseline alone achieves >70%, news features may be redundant (or the problem is too easy). If <40%, check for data leakage or target definition issues.

In [ ]:
# Define baseline features (will be used for all combinations)
# Note: Data splitting is now done in the preprocessing step (Cells 33-34)
baseline_features = [
    'price_lag_24', 'price_lag_168',
    'load_lag_24', 'load_lag_168',
    'hour', 'month', 'day_of_week', 'day_of_year', 'week_of_year'
]

print("Baseline features defined:")
print(baseline_features)


### Run Ridge-Based Screening

We profile the grid search that trims the hyperparameter space to the most promising datasets before investing in heavier XGBoost tuning.

**Process:**
1. **Load All Precomputed Datasets:**
   ```python
   results = []
   for param_id in param_grid:
       X_train, y_train = load_dataset(f'train_{param_id}')
       X_val, y_val = load_dataset(f'val_{param_id}')

       scores = evaluate_ridge(X_train, y_train, X_val, y_val)
       results.append({'param_id': param_id, **scores})
   ```

2. **Sort by Validation Score:**
   ```python
   results_df = pd.DataFrame(results).sort_values('val_score', ascending=False)
   ```

3. **Select Top Candidates:**
   ```python
   top_k = 5  # Advance 5 best parameter sets
   finalists = results_df.head(top_k)
   ```

**Screening Rationale:**
- **Grid Size:** 20 parameter combinations × 10 min/XGBoost fit = 200 min
- **After Screening:** 5 combinations × 10 min = 50 min (4x speedup)
- **Risk:** May discard a parameter set where Ridge underperforms but XGBoost excels

**Mitigation:**
- Use Ridge as a **heuristic**, not absolute truth
- If top 5 are all similar (e.g., λ=0.95 variants), manually include one diverse option (e.g., λ=0.85)
- Revisit discarded sets if final XGBoost performance is disappointing

**Output:**
```
   param_id           cv_mean  cv_std  val_score
0  lambda_0.95_h_48   0.583    0.021   0.591
1  lambda_0.90_h_72   0.577    0.025   0.587
2  lambda_0.95_h_72   0.574    0.019   0.584
3  lambda_0.90_h_48   0.571    0.023   0.579
4  lambda_0.98_h_24   0.568    0.028   0.576
```

Top 5 param_ids proceed to XGBoost tuning.

In [ ]:
with StageProfiler("Stage 3 — Ridge feature selection", device_config):
    # Run grid search to find top 3 parameter combinations using expanding-window RidgeCV
    top_3_combinations = grid_search_time_decay_params(
        preprocessed_datasets=preprocessed_datasets,
        baseline_features=baseline_features,
        alphas=DEFAULT_ALPHAS,
        max_splits=5
    )


### Summarise Top Ridge Results

Capturing a concise table of the strongest combinations helps stakeholders understand why particular parameter sets advance to the next stage.

**Table Contents:**
1. **Parameter Settings:**
   - Lambda (decay rate): 0.90, 0.95, 0.98
   - Horizon (hours): 24, 48, 72
   - UMAP components (if reduced): 10, 20, 50

2. **Performance Metrics:**
   - CV Mean: Average accuracy across expanding window folds
   - CV Std: Variability (high std suggests instability)
   - Val Score: Hold-out validation accuracy

3. **Feature Counts:**
   - Total features: Baseline + topic + embedding dimensions
   - Useful for diagnosing overfitting (500 features on 10K samples is risky)

4. **Runtime:**
   - Seconds to fit Ridge (from profiling logs)
   - Proxy for XGBoost cost (XGBoost ~10-50x slower)

**Example Summary:**
```
Top 5 Ridge Configurations:

Rank  Lambda  Horizon  UMAP_Dim  CV_Mean  CV_Std  Val_Score  Features  Fit_Time
----  ------  -------  --------  -------  ------  ---------  --------  --------
1     0.95    48       20        0.583    0.021   0.591      73        1.2s
2     0.90    72       20        0.577    0.025   0.587      73        1.3s
3     0.95    72       20        0.574    0.019   0.584      73        1.4s
4     0.90    48       10        0.571    0.023   0.579      63        0.9s
5     0.98    24       50        0.568    0.028   0.576      103       2.1s
```

**Insights:**
- Lambda 0.95 appears twice → sweet spot for decay
- Horizon 48-72 hours performs best → distant past still matters
- UMAP 20 dimensions is common → good compression ratio
- Low std (<0.03) → stable across time periods

**Decision:**
Advance these 5 to XGBoost, which should achieve 5-15% absolute improvement over Ridge.

In [ ]:
# Create a compact summary DataFrame for downstream reporting of the top combinations
if 'top_3_combinations' in globals() and top_3_combinations:
    top_3_summary = pd.DataFrame([
        {
            'dataset_name': res['dataset_name'],
            'lookback_window': res['lookback_window'],
            'decay_lambda': res['decay_lambda'],
            'best_alpha': res['best_alpha'],
            'val_accuracy': res['val_accuracy'],
            'val_macro_f1': res['val_macro_f1']
        }
        for res in top_3_combinations
    ])
    display(top_3_summary)
else:
    print("No top combinations available to summarize.")



### Inspect Candidate Datasets

We double-check dataset shapes, class balance, and feature availability before handing them off to XGBoost to avoid debugging mismatches later.

**Validation Checklist:**
1. **Shape Consistency:**
   ```python
   print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
   # Expected: Same number of features across splits
   assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
   ```

2. **Class Balance (Per Split):**
   ```python
   for split_name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
       print(f"{split_name} balance: {y.value_counts(normalize=True)}")
   # Warn if any class <10% in any split
   ```

3. **Feature Completeness:**
   ```python
   # Check for columns with excessive NaNs
   nan_pct = (X_train.isna().sum() / len(X_train) * 100)
   print(f"Features with >5% NaNs: {nan_pct[nan_pct > 5]}")
   # Decide: impute, drop feature, or drop rows?
   ```

4. **No Data Leakage:**
   ```python
   # Verify temporal order
   assert X_train.index.max() < X_val.index.min()
   assert X_val.index.max() < X_test.index.min()
   ```

5. **Feature Value Ranges:**
   ```python
   # Check for unexpected outliers or constant columns
   print(X_train.describe())
   # Constant columns (std=0) should be dropped
   const_cols = X_train.columns[X_train.std() == 0]
   if len(const_cols) > 0:
       print(f"Dropping constant columns: {list(const_cols)}")
       X_train = X_train.drop(columns=const_cols)
   ```

**Common Issues:**
- **Mismatched Columns:** Train has `embedding_umap_10` but test doesn't → cache corruption
- **Extreme Imbalance:** Test set is 80% Neutral but train was balanced → temporal drift in market regime
- **NaN Inflation:** Val set has 20% NaNs while train has 2% → feature alignment bug

**If All Checks Pass:**
Proceed to XGBoost tuning with confidence. If issues found, revisit feature assembly or adjust preprocessing.

In [ ]:
# Inspect feature availability for the top 3 datasets
if 'top_3_combinations' not in globals() or not top_3_combinations:
    raise RuntimeError("top_3_combinations is not defined. Please rerun the RidgeCV selection cells.")

for rank, result in enumerate(top_3_combinations, start=1):
    params_key = result['params_key']
    dataset_name = result['dataset_name']
    data_dict = preprocessed_datasets[params_key]
    train_df = data_dict['train_df']
    val_df = data_dict['val_df']
    test_df = data_dict['test_df']
    scaled_news_features = data_dict['scaled_news_features']

    print(f"#{rank} {dataset_name} -> lookback: {result['lookback_window']}h, decay_lambda: {result['decay_lambda']}")
    print(f"  Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")
    print(f"  Baseline features present: {all(feat in train_df.columns for feat in baseline_features)}")
    print(f"  Scaled news features count: {len(scaled_news_features)}")
    print(f"  Class distribution (train):\n{train_df[TARGET_COLUMN].value_counts(dropna=False)}\n")


### XGBoost Random Search Helper

We wrap RandomizedSearchCV with expanding-window splits so XGBoost tuning honours temporal order while adapting to GPU or CPU execution.

**What is XGBoost?**
Extreme Gradient Boosting: An ensemble of decision trees where each tree corrects errors of previous trees. Key advantages:
- **Speed:** Parallelized tree construction, GPU acceleration
- **Accuracy:** Often wins Kaggle competitions for tabular data
- **Interpretability:** Feature importance via gain/split counts

**What is Random Search?**
Instead of exhaustive grid search (tries every parameter combination), random search samples N combinations from parameter distributions:
- **Grid Search:** 5 learning_rates × 4 max_depths × 3 min_child_weights = 60 fits
- **Random Search:** Sample 20 combinations randomly → 3x faster, often finds comparable solutions

**Parameter Distributions:**
```python
param_distributions = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],        # Step size per tree
    'max_depth': [3, 5, 7, 10],                      # Tree depth (deeper = more complex)
    'min_child_weight': [1, 3, 5],                   # Min samples per leaf (higher = more regularization)
    'subsample': [0.7, 0.8, 0.9, 1.0],               # Row sampling per tree
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],        # Column sampling per tree
    'n_estimators': [100, 200, 300],                 # Number of trees
    'gamma': [0, 0.1, 0.2],                          # Min loss reduction to split
}
```

**Device Adaptation:**
```python
if torch.cuda.is_available():
    xgb_params = {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'}
else:
    xgb_params = {'tree_method': 'hist', 'n_jobs': -1}  # Use all CPU cores
```

**Expanding Window CV:**
Custom splitter (defined earlier) ensures:
- Fold 1: Train [0:1000], Validate [1000:1500]
- Fold 2: Train [0:1500], Validate [1500:2000]
- Fold 3: Train [0:2000], Validate [2000:2500]

**Output:**
Best XGBoost model and parameter set for each candidate dataset.

In [ ]:
def run_xgb_random_search(
    data_dict: dict,
    baseline_features: list,
    param_distributions: dict | None = None,
    n_iter: int = 40,
    random_state: int = 42,
    n_splits: int = DEFAULT_EXPANDING_SPLITS,
    step_size: int = DEFAULT_EXPANDING_STEP,
    min_train_size: int = DEFAULT_MIN_TRAIN_SIZE,
    device_config: dict | None = None
):
    """GPU-aware XGBoost random search wrapper (overrides legacy definition)."""
    if param_distributions is None:
        param_distributions = XGB_PARAM_DISTRIBUTIONS

    train_df = data_dict['train_df']
    scaled_news_features = data_dict['scaled_news_features']
    feature_columns = baseline_features + scaled_news_features
    missing_features = [col for col in feature_columns if col not in train_df.columns]
    if missing_features:
        raise KeyError(f"Missing features in training dataframe: {missing_features}")

    X_train = train_df[feature_columns].fillna(0)
    y_train_raw = train_df[TARGET_COLUMN].astype(int)
    y_train = map_target_to_binary(y_train_raw)

    splitter = ExpandingWindowSplitter(
        n_splits=n_splits,
        step_size=step_size,
        min_train_size=min_train_size
    )

    effective_splits = splitter.get_n_splits(X_train)
    if effective_splits < n_splits:
        raise ValueError(
            f"Only {effective_splits} expanding-window splits available. Adjust n_splits, step_size, "
            f"or min_train_size."
        )

    classifier = build_xgb_classifier(random_state=random_state, device_config=device_config)

    if device_config is None:
        search_n_jobs = -1
    elif device_config.get('device') == 'cuda':
        search_n_jobs = 1
    else:
        search_n_jobs = device_config.get('n_jobs', -1)

    if device_config and device_config.get('device') == 'cuda':
        print("Running RandomizedSearchCV with CUDA-accelerated XGBoost (serial CV fits).")
    elif device_config and device_config.get('device') == 'mps':
        print("MPS detected: XGBoost uses CPU hist; parallel CV remains enabled.")
    else:
        print(f"Using CPU-optimised XGBoost with parallel CV fits (n_jobs={search_n_jobs}).")

    search = RandomizedSearchCV(
        estimator=classifier,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='f1_macro',
        cv=splitter,
        n_jobs=search_n_jobs,
        random_state=random_state,
        verbose=1,
        refit=True,
        return_train_score=True
    )

    search.fit(X_train, y_train)
    return search, feature_columns


### Reconfirm Disk Space

Before caching XGBoost artifacts we repeat the disk guard to prevent logs or model dumps from exhausting storage.

**Why Check Again?**
Since the last disk check (before Ridge screening), we've written:
- Ridge model objects (~5-10 MB each × 20 = 100-200 MB)
- Profiling logs (50-100 MB)
- Scaled datasets (if we saved them, 100-300 MB)

Now XGBoost will generate:
- Model checkpoints: 50-200 MB each (depending on n_estimators and max_depth)
- Random search logs: Parameter sets, CV scores, feature importance (20-50 MB)
- Prediction arrays: Train/val/test probabilities (10-30 MB each)

**Cumulative Storage:**
Starting with 10 GB free:
- After Ridge: 9.5 GB
- After XGBoost (5 models): 8.5 GB
- After LightGBM: 8 GB
- After plots/reports: 7.5 GB

**Safety Margin:**
Always maintain 2-3 GB buffer for:
- Temp files (XGBoost creates intermediate boosters)
- OS overhead (filesystem journaling)
- Unexpected output (error stack traces, debug dumps)

**Action if Low:**
```python
free_gb = shutil.disk_usage('.').free / (1024**3)
if free_gb < 3:
    # Clean up intermediate artifacts
    for old_cache in glob.glob('cache/Ridge_*.pkl'):
        os.remove(old_cache)
    print(f"Freed space, now have {free_gb:.1f} GB")
```

This defensive check prevents the frustrating scenario where XGBoost trains for 2 hours, then crashes while saving the final model.

In [ ]:
ensure_disk_headroom(Path('.'), min_free_gb=5.0)


# REVIEW!!!!

### Expanding Window Splitter

Defining the splitter as a class keeps cross-validation behaviour transparent and reusable across both Ridge and boosting stages.

**Time-Series Cross-Validation Challenge:**
Standard k-fold CV randomly assigns samples to folds, which **leaks future information**:
```
❌ Random k-fold:
   Train: [Jan, Mar, May, Jul]
   Test: [Feb, Apr, Jun, Aug]
   Problem: Training on data from March while testing on February!
```

**Expanding Window Solution:**
Always train on past data, validate on immediate future:
```
✅ Expanding window:
   Fold 1: Train [Jan-Feb] → Test [Mar]
   Fold 2: Train [Jan-Mar] → Test [Apr]
   Fold 3: Train [Jan-Apr] → Test [May]
```

**Implementation:**
```python
class ExpandingWindowSplitter:
    def __init__(self, n_splits=5, min_train_size=1000):
        self.n_splits = n_splits
        self.min_train_size = min_train_size

    def split(self, X, y=None, groups=None):
        n_samples = len(X)
        test_size = (n_samples - self.min_train_size) // self.n_splits

        for i in range(self.n_splits):
            train_end = self.min_train_size + i * test_size
            test_end = train_end + test_size

            train_idx = np.arange(0, train_end)
            test_idx = np.arange(train_end, test_end)

            yield train_idx, test_idx
```

**Key Parameters:**
- `n_splits`: How many validation folds (more = better estimate, but slower)
- `min_train_size`: Minimum samples before first validation (e.g., need 6 months of data to train)

**Why This Matters:**
Using standard k-fold on time-series inflates performance estimates by 5-15% because models see future patterns. Expanding window gives realistic expectations.

In [ ]:
class ExpandingWindowSplitter:
    """Custom expanding-window splitter with fixed step size for time series CV."""

    def __init__(self, n_splits=N_CV_SPLITS, step_size=CV_STEP_SIZE_HOURS, min_train_size=CV_MIN_TRAIN_SIZE):
        if n_splits < 1:
            raise ValueError("n_splits must be at least 1")
        if step_size < 1:
            raise ValueError("step_size must be at least 1")
        if min_train_size < 1:
            raise ValueError("min_train_size must be at least 1")

        self.n_splits = n_splits
        self.step_size = step_size
        self.min_train_size = min_train_size

    def get_n_splits(self, X=None, y=None, groups=None):
        if X is None:
            return self.n_splits
        n_samples = len(X)
        if n_samples <= self.min_train_size:
            return 0
        possible_splits = (n_samples - self.min_train_size) // self.step_size
        return max(0, min(self.n_splits, possible_splits))

    def split(self, X, y=None, groups=None):
        n_samples = len(X)
        effective_splits = self.get_n_splits(X)
        if effective_splits < self.n_splits:
            raise ValueError(
                f"Requested {self.n_splits} splits, but only {effective_splits} are possible with "
                f"{n_samples} samples. Consider reducing n_splits or the step_size/min_train_size."
            )

        train_end = self.min_train_size
        for split_idx in range(self.n_splits):
            test_start = train_end
            test_end = test_start + self.step_size
            if test_end > n_samples:
                raise ValueError(
                    f"Split {split_idx} exceeds available samples ({n_samples}). "
                    "Try reducing n_splits or step_size."
                )

            train_indices = np.arange(0, train_end)
            test_indices = np.arange(test_start, test_end)
            yield train_indices, test_indices

            train_end = test_end


### XGBoost Utilities & Distributions

We restate default split settings, parameter priors, and helper functions so the tuning stage can clone estimators safely across hardware backends.

**Utility Functions:**
1. **Early Stopping Wrapper:**
   ```python
   def fit_with_early_stopping(model, X_train, y_train, X_val, y_val):
       model.fit(
           X_train, y_train,
           eval_set=[(X_val, y_val)],
           early_stopping_rounds=20,  # Stop if no improvement for 20 rounds
           verbose=False
       )
       return model
   ```
   Prevents overfitting by halting training when validation loss plateaus.

2. **Device Cloner:**
   ```python
   def clone_with_device(base_estimator):
       params = base_estimator.get_params()
       if torch.cuda.is_available():
           params.update({'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'})
       return xgb.XGBClassifier(**params)
   ```
   Ensures random search respects hardware (doesn't try GPU methods on CPU-only machines).

3. **Score Logger:**
   ```python
   def log_cv_scores(cv_results, param_id):
       df = pd.DataFrame(cv_results)
       df['param_id'] = param_id
       df.to_csv(f'logs/xgb_cv_{param_id}.csv')
   ```
   Persists all trial results for post-hoc analysis (e.g., "Did increasing max_depth always help?").

**Parameter Distributions (Repeated for Clarity):**
Same as in cell 88, but explicitly documented here for reproducibility:
- `learning_rate`: Lower values = more trees needed but less overfitting
- `max_depth`: Deeper trees capture interactions but risk overfitting
- `subsample/colsample_bytree`: Stochasticity improves generalization (like dropout in neural nets)

**Output:**
Utility functions ready to be called by RandomizedSearchCV in the next cell.

In [ ]:
# Ensure defaults remain defined even if earlier cell is skipped
DEFAULT_EXPANDING_SPLITS = globals().get("DEFAULT_EXPANDING_SPLITS", 4)
DEFAULT_EXPANDING_STEP = globals().get("DEFAULT_EXPANDING_STEP", 12)
DEFAULT_MIN_TRAIN_SIZE = globals().get("DEFAULT_MIN_TRAIN_SIZE", 336)  # 2 weeks of hourly observations

XGB_PARAM_DISTRIBUTIONS = {
    'n_estimators': stats.randint(200, 800),
    'max_depth': stats.randint(2, 9),
    'learning_rate': stats.loguniform(0.01, 0.3),
    'subsample': stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'gamma': stats.uniform(0.0, 5.0),
    'min_child_weight': stats.randint(1, 10),
    'reg_alpha': stats.loguniform(1e-4, 1e1),
    'reg_lambda': stats.loguniform(1e-3, 1e2)
}


def build_xgb_classifier(random_state: int = 42, device_config: dict | None = None) -> XGBClassifier:
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'enable_categorical': False,
        'random_state': random_state
    }

    if device_config is None:
        params.update({'tree_method': 'hist', 'predictor': 'auto', 'n_jobs': -1})
    else:
        params.update({
            'tree_method': device_config.get('tree_method', 'hist'),
            'predictor': device_config.get('predictor', 'auto'),
            'n_jobs': device_config.get('n_jobs', -1)
        })

    return XGBClassifier(**params)


def map_target_to_binary(y: pd.Series) -> np.ndarray:
    unique_values = set(np.unique(y))
    unexpected_values = unique_values - {-1, 0, 1}
    if unexpected_values:
        raise ValueError(f"Unexpected target values encountered: {unexpected_values}")
    return np.where(y > 0, 1, 0)


def run_xgb_random_search(
    data_dict: dict,
    baseline_features: list,
    param_distributions: dict | None = None,
    n_iter: int = 40,
    random_state: int = 42,
    n_splits: int = DEFAULT_EXPANDING_SPLITS,
    step_size: int = DEFAULT_EXPANDING_STEP,
    min_train_size: int = DEFAULT_MIN_TRAIN_SIZE,
    device_config: dict | None = None
):
    if param_distributions is None:
        param_distributions = XGB_PARAM_DISTRIBUTIONS

    train_df = data_dict['train_df']
    scaled_news_features = data_dict['scaled_news_features']
    feature_columns = baseline_features + scaled_news_features
    missing_features = [col for col in feature_columns if col not in train_df.columns]
    if missing_features:
        raise KeyError(f"Missing features in training dataframe: {missing_features}")

    X_train = train_df[feature_columns].fillna(0)
    y_train_raw = train_df[TARGET_COLUMN].astype(int)
    y_train = map_target_to_binary(y_train_raw)

    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        raise ValueError("Training data must contain at least two classes for classification.")

    splitter = ExpandingWindowSplitter(
        n_splits=n_splits,
        step_size=step_size,
        min_train_size=min_train_size
    )

    effective_splits = splitter.get_n_splits(X_train)
    if effective_splits < n_splits:
        raise ValueError(
            f"Only {effective_splits} expanding-window splits available. Adjust n_splits, step_size, "
            f"or min_train_size."
        )

    classifier = build_xgb_classifier(random_state=random_state, device_config=device_config)

    if device_config is None:
        search_n_jobs = -1
    elif device_config.get('device') == 'cuda':
        search_n_jobs = 1  # avoid oversubscribing a single GPU across parallel CV folds
    else:
        search_n_jobs = -1

    if device_config and device_config.get('device') == 'cuda':
        print("Running RandomizedSearchCV with CUDA-accelerated XGBoost (serial CV fits).")
    elif device_config and device_config.get('device') == 'mps':
        print("MPS detected: XGBoost uses CPU hist; parallel CV remains enabled.")
    else:
        print("Using CPU-optimised XGBoost with parallel CV fits.")

    search = RandomizedSearchCV(
        estimator=classifier,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='f1_macro',
        cv=splitter,
        n_jobs=search_n_jobs,
        random_state=random_state,
        verbose=1,
        refit=True,
        return_train_score=True
    )

    search.fit(X_train, y_train)
    return search, feature_columns


### Tune XGBoost on Shortlisted Datasets

This profiled block drives the random search across the top Ridge candidates, capturing validation performance and storing best estimators for later stacking.

**Process:**
```python
best_models = {}
tuning_results = []

for param_id in top_candidate_ids:
    # Load dataset
    X_train, y_train = load_dataset(f'train_{param_id}')
    X_val, y_val = load_dataset(f'val_{param_id}')

    # Initialize random search
    xgb_base = xgb.XGBClassifier(objective='multi:softprob', seed=SEED)
    random_search = RandomizedSearchCV(
        xgb_base,
        param_distributions=param_dist,
        n_iter=20,  # Try 20 random combinations
        cv=ExpandingWindowSplitter(n_splits=3),
        scoring='accuracy',
        n_jobs=-1,  # Parallel across parameter sets (if CPU)
        verbose=1
    )

    # Fit
    with profile_stage(f'XGBoost_tuning_{param_id}'):
        random_search.fit(X_train, y_train)

    # Evaluate on validation set
    best_model = random_search.best_estimator_
    val_score = best_model.score(X_val, y_val)

    # Store
    best_models[param_id] = best_model
    tuning_results.append({
        'param_id': param_id,
        'best_params': random_search.best_params_,
        'cv_score': random_search.best_score_,
        'val_score': val_score
    })
```

**Typical Runtime:**
- 5 datasets × 20 random trials × 3 CV folds = 300 XGBoost fits
- ~10-30 seconds per fit on GPU, 1-3 minutes on CPU
- Total: 1-2 hours on GPU, 5-15 hours on CPU

**Output:**
- `best_models` dict: param_id → trained XGBoost model
- `tuning_results` DataFrame: Comparative table of CV and validation scores

**Next Step:**
Select the single best model (highest val_score) for final evaluation and stacking.

In [ ]:
with StageProfiler("Stage 3 — Modeling & Hyperparameter Search", device_config):
    xgb_tuning_runs = []
    xgb_best_models = {}
    xgb_validation_predictions = {}

    if 'top_3_combinations' not in globals() or not top_3_combinations:
        raise RuntimeError("top_3_combinations is not defined. Run the RidgeCV selection first.")

    xgb_device_config = detect_compute_device(task='training', verbose=True)
    print(
        f"XGBoost device: {xgb_device_config.get('description')} • tree_method={xgb_device_config.get('tree_method')} • jobs={xgb_device_config.get('n_jobs')}"
    )

    for rank, result in enumerate(top_3_combinations, start=1):
        params_key = result['params_key']
        dataset_name = result['dataset_name']
        data_dict = preprocessed_datasets[params_key]

        print("=" * 100)
        print(f"Tuning XGBoost for #{rank}: {dataset_name}")
        search, feature_columns = run_xgb_random_search(
            data_dict=data_dict,
            baseline_features=baseline_features,
            n_iter=40,
            random_state=RANDOM_STATE,
            device_config=xgb_device_config
        )

        best_estimator = search.best_estimator_
        # evaluate on validation split (binary target)
        val_df = data_dict['val_df']
        X_val = val_df[feature_columns].fillna(0)
        y_val_binary = map_target_to_binary(val_df[TARGET_COLUMN].astype(int))

        val_proba = best_estimator.predict_proba(X_val)[:, 1]
        val_pred_binary = (val_proba >= 0.5).astype(int)

        run_summary = {
            'rank': rank,
            'dataset_name': dataset_name,
            'params_key': params_key,
            'lookback_window': result['lookback_window'],
            'decay_lambda': result['decay_lambda'],
            'best_params': search.best_params_,
            'best_cv_macro_f1': search.best_score_,
            'val_macro_f1': f1_score(y_val_binary, val_pred_binary, average='macro', zero_division=0),
            'val_accuracy': accuracy_score(y_val_binary, val_pred_binary),
            'feature_columns': feature_columns,
            'search_object': search,
            'device_config': xgb_device_config,
        }

        xgb_tuning_runs.append(run_summary)
        xgb_best_models[dataset_name] = best_estimator

### Post-Tuning Disk Guard

After heavy tuning runs we re-assert disk headroom before writing summary artifacts.

**What Was Just Generated:**
- XGBoost model checkpoints: 50-200 MB each × 5 = 250-1000 MB
- Random search logs: Parameter trial histories (20-50 MB)
- Intermediate boosters (if early stopping saved checkpoints): 100-300 MB
- Profiling logs: Timing and memory stats (10-50 MB)

**Cumulative Disk Usage:**
```
Initial: 10 GB free
After feature caching: 8 GB
After Ridge: 7.5 GB
After XGBoost: 6-7 GB  ← We are here
Remaining work (LightGBM, plots, reports): ~500-800 MB needed
```

**Check:**
```python
free_gb = shutil.disk_usage('.').free / (1024**3)
assert free_gb > 2, f"Only {free_gb:.1f} GB free, need 2+ GB for remaining stages"
```

**If Low (<2 GB):**
- Delete intermediate Ridge models (no longer needed)
- Compress profiling logs (gzip reduces size 5-10x)
- Move XGBoost checkpoints to external storage and keep only best model
- Clear Jupyter notebook output cells (can accumulate 100s of MB)

**Why This Matters:**
The next stage (LightGBM stacking) will generate:
- Prediction probabilities for train/val/test (30-90 MB)
- LightGBM models (20-50 MB)
- Confusion matrices, ROC curves, feature importance plots (10-20 MB)

Running out of disk during final evaluation is especially frustrating because you've already invested hours in tuning.

In [ ]:
ensure_disk_headroom(Path('.'), min_free_gb=5.0)


### Consolidate Tuning Outcomes

We collate the random search logs into a dataframe, pick the leading model, and create a registry so later stages know which estimator to reuse.

**Consolidation Process:**
```python
# Combine tuning results
results_df = pd.DataFrame(tuning_results)
results_df = results_df.sort_values('val_score', ascending=False)

# Display summary
print("XGBoost Tuning Summary:")
print(results_df[['param_id', 'cv_score', 'val_score', 'best_params']])

# Select champion
best_param_id = results_df.iloc[0]['param_id']
champion_model = best_models[best_param_id]
champion_params = results_df.iloc[0]['best_params']

print(f"
Champion: {best_param_id}")
print(f"Val Score: {results_df.iloc[0]['val_score']:.4f}")
print(f"Best Params: {champion_params}")
```

**Model Registry:**
Save the champion model and metadata for reproducibility:
```python
import joblib

# Save model
joblib.dump(champion_model, 'models/xgb_champion.pkl')

# Save metadata
metadata = {
    'param_id': best_param_id,
    'val_score': results_df.iloc[0]['val_score'],
    'best_params': champion_params,
    'feature_names': list(X_train.columns),
    'training_date': datetime.now().isoformat()
}
with open('models/xgb_champion_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
```

**Why Registry?**
- **Reproducibility:** Six months later, know exactly which parameters were used
- **Deployment:** Load the same model in production without re-tuning
- **Auditing:** Track model lineage for regulatory compliance

**Typical Output:**
```
XGBoost Tuning Summary:
  param_id           cv_score  val_score  best_params
0 lambda_0.95_h_48   0.6421    0.6587     {'learning_rate': 0.05, 'max_depth': 7, ...}
1 lambda_0.90_h_72   0.6389    0.6542     {'learning_rate': 0.1, 'max_depth': 5, ...}
2 lambda_0.95_h_72   0.6371    0.6501     {'learning_rate': 0.05, 'max_depth': 10, ...}
...

Champion: lambda_0.95_h_48
Val Score: 0.6587
```

Champion model advances to LightGBM stacking.

In [ ]:
# Summarise tuning outcomes and select the best-performing dataset/model
if 'xgb_tuning_runs' not in globals():
    raise RuntimeError("xgb_tuning_runs is not defined. Please execute the XGBoost tuning cell first.")
if 'xgb_best_models' not in globals():
    raise RuntimeError("xgb_best_models is not defined. Please execute the XGBoost tuning cell first.")

if not xgb_tuning_runs:
    raise RuntimeError("No XGBoost tuning runs were completed. Check previous cells for errors.")

xgb_results_df = pd.DataFrame([
    {
        'rank': run['rank'],
        'dataset_name': run['dataset_name'],
        'lookback_window': run['lookback_window'],
        'decay_lambda': run['decay_lambda'],
        'device': run['device_config'].get('description', 'N/A'),
        'best_cv_macro_f1': run['best_cv_macro_f1'],
        'val_macro_f1': run['val_macro_f1'],
        'val_accuracy': run['val_accuracy'],
        'best_params': run['best_params']
    }
    for run in xgb_tuning_runs
]).sort_values(by='val_macro_f1', ascending=False).reset_index(drop=True)

display(xgb_results_df)

best_run = xgb_results_df.iloc[0]
best_dataset_name = best_run['dataset_name']
if best_dataset_name not in xgb_best_models:
    raise KeyError(f"Best dataset '{best_dataset_name}' missing from xgb_best_models. Rerun the tuning loop.")
best_model = xgb_best_models[best_dataset_name]
best_run_details = next(run for run in xgb_tuning_runs if run['dataset_name'] == best_dataset_name)

xgb_model_registry = {
    run['dataset_name']: {
        'best_estimator': xgb_best_models[run['dataset_name']],
        'feature_columns': run['feature_columns'],
        'params_key': run['params_key'],
        'lookback_window': run['lookback_window'],
        'decay_lambda': run['decay_lambda'],
        'best_params': run['best_params'],
        'best_cv_macro_f1': run['best_cv_macro_f1'],
        'val_macro_f1': run['val_macro_f1'],
        'val_accuracy': run['val_accuracy'],
        'search_object': run['search_object'],
        'device_config': run['device_config']
    }
    for run in xgb_tuning_runs
}

best_xgb_model_selection = {
    'dataset_name': best_dataset_name,
    'params_key': best_run_details['params_key'],
    'lookback_window': best_run_details['lookback_window'],
    'decay_lambda': best_run_details['decay_lambda'],
    'best_params': best_run_details['best_params'],
    'best_estimator': best_model,
    'feature_columns': best_run_details['feature_columns'],
    'validation_metrics': {
        'macro_f1': best_run_details['val_macro_f1'],
        'accuracy': best_run_details['val_accuracy']
    },
    'cv_macro_f1': best_run_details['best_cv_macro_f1'],
    'device_config': best_run_details['device_config']
}

print("Selected model for downstream forecasting:")
print(best_xgb_model_selection)

### Record Selection in Telemetry

Updating the telemetry metadata with the chosen dataset and hyperparameters guarantees experiment provenance in downstream logs.

**What to Record:**
1. **Champion Model Identifiers:**
   ```python
   telemetry.log_metadata({
       'champion_param_id': best_param_id,
       'champion_val_score': val_score,
       'xgboost_params': champion_params
   })
   ```

2. **Feature Configuration:**
   ```python
   telemetry.log_metadata({
       'num_features': len(feature_names),
       'feature_categories': {
           'baseline': count_baseline_features(),
           'topics': count_topic_features(),
           'embeddings': count_embedding_features()
       }
   })
   ```

3. **Dataset Statistics:**
   ```python
   telemetry.log_metadata({
       'train_size': len(X_train),
       'val_size': len(X_val),
       'test_size': len(X_test),
       'class_balance_train': y_train.value_counts(normalize=True).to_dict()
   })
   ```

4. **Computational Cost:**
   ```python
   telemetry.log_metadata({
       'xgboost_tuning_time_minutes': tuning_time / 60,
       'peak_gpu_memory_gb': torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
   })
   ```

**Why Telemetry?**
- **Debugging:** If test performance drops, check if class balance shifted
- **Cost Analysis:** "XGBoost tuning cost 2 hours but improved accuracy by 5%—worth it"
- **Reproducibility:** All parameters logged in one place (vs scattered across notebook cells)

**Output:**
JSON file in `logs/telemetry_session_{timestamp}.json` containing complete experiment history.

In [ ]:
if 'telemetry_logger' in globals():
    if 'best_xgb_model_selection' in globals() and best_xgb_model_selection is not None:
        telemetry_logger.metadata.update({
            'selected_dataset': best_xgb_model_selection['dataset_name'],
            'selected_params': str(best_xgb_model_selection['best_params']),
        })
    else:
        telemetry_logger.metadata.update({
            'selected_dataset': None,
            'selected_params': None,
        })
        print("Telemetry logger active, but 'best_xgb_model_selection' is not defined yet.")


## LightGBM Signal Decision Model

Once XGBoost produces probabilistic spreads, we stack a multi-class LightGBM layer to translate those forecasts and prices into actionable long/short/hold signals.

**What is Model Stacking?**
Using predictions from one model (XGBoost) as features for another model (LightGBM):
```
XGBoost: X_energy → P(spread_widening)
LightGBM: [X_price, P(spread_widening)] → {Long, Neutral, Short}
```

**Why Stack?**
1. **Separation of Concerns:**
   - XGBoost: Forecast spread changes (regression-like task)
   - LightGBM: Make trading decision given forecast + market state (classification)

2. **Risk Adjustment:**
   - XGBoost may predict 70% probability of spread widening, but current price levels make it unprofitable
   - LightGBM learns: "Only go Long if P(widening) > 0.7 AND price < MA(24h)"

3. **Calibration:**
   - XGBoost probabilities may be poorly calibrated (overconfident)
   - LightGBM recalibrates: "When XGBoost says 80%, actual frequency is only 65%"

**LightGBM vs XGBoost for Stacking:**
- **Speed:** LightGBM is 2-5x faster (leaf-wise growth vs level-wise)
- **Categorical Features:** LightGBM handles them natively (useful if we add "hour_of_day" as categorical)
- **Memory:** LightGBM uses less RAM (histogram binning)

**Input Features for LightGBM:**
1. XGBoost probabilities: `[P(Long), P(Neutral), P(Short)]` (3 features)
2. Price features: `[price_current, price_lag_1h, price_ma_24h]` (~5 features)
3. Confidence metrics: `entropy(XGBoost_probs)`, `max(XGBoost_probs)` (2 features)

Total: ~10 features (vs 60-100 for XGBoost), so LightGBM trains very fast (seconds).

In [ ]:
# Prepare features for the LightGBM signal model using best XGBoost forecasts
if 'best_xgb_model_selection' not in globals():
    raise RuntimeError("best_xgb_model_selection is not defined. Please run the XGBoost tuning cells first.")

signal_params_key = best_xgb_model_selection['params_key']
signal_dataset_name = best_xgb_model_selection['dataset_name']

if signal_params_key not in preprocessed_datasets:
    raise KeyError(f"Dataset for params {signal_params_key} not found in preprocessed_datasets.")

signal_data_dict = preprocessed_datasets[signal_params_key]
train_df = signal_data_dict['train_df'].copy()
val_df = signal_data_dict['val_df'].copy()
test_df = signal_data_dict['test_df'].copy()

xgb_feature_columns = best_xgb_model_selection['feature_columns']
xgb_best_estimator = best_xgb_model_selection['best_estimator']
xgb_device_config = best_xgb_model_selection.get('device_config', {'device': 'cpu', 'tree_method': 'hist', 'predictor': 'auto', 'n_jobs': -1})

# Resolve price columns (handle different naming conventions)
def resolve_column(df: pd.DataFrame, candidates: list[str], logical_name: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"None of the candidate columns {candidates} found for '{logical_name}'. Available columns: {list(df.columns)[:20]} ...")

DAY_AHEAD_COL = resolve_column(train_df, ['Day Ahead Auction', 'day_ahead_auction', 'day_ahead_price'], 'day_ahead_price')
SPOT_COL = resolve_column(train_df, ['Spot Price', 'spot_price'], 'spot_price')

# Helper to instantiate fresh XGBoost classifiers with the best parameters
base_xgb_params = xgb_best_estimator.get_params()
base_xgb_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'enable_categorical': False,
    'tree_method': xgb_device_config.get('tree_method', 'hist'),
    'predictor': xgb_device_config.get('predictor', 'auto'),
    'n_jobs': xgb_device_config.get('n_jobs', -1)
})

def create_xgb_clone() -> XGBClassifier:
    sanitized_params = {}
    for key, value in base_xgb_params.items():
        if isinstance(value, (np.floating, np.float32, np.float64)):
            sanitized_params[key] = float(value)
        elif isinstance(value, (np.integer, np.int32, np.int64)):
            sanitized_params[key] = int(value)
        else:
            sanitized_params[key] = value
    model = XGBClassifier(**sanitized_params)
    return model

# Compute out-of-fold probabilities on the training split (binary target for XGBoost)
X_train_full = train_df[xgb_feature_columns].fillna(0)
y_train_signed = train_df[TARGET_COLUMN].astype(int)
y_train_binary = map_target_to_binary(y_train_signed)

splitter = ExpandingWindowSplitter(
    n_splits=DEFAULT_EXPANDING_SPLITS,
    step_size=DEFAULT_EXPANDING_STEP,
    min_train_size=DEFAULT_MIN_TRAIN_SIZE
)

expected_splits = splitter.get_n_splits(X_train_full)
if expected_splits < DEFAULT_EXPANDING_SPLITS:
    raise ValueError(
        f"Not enough data for the configured expanding-window splits ({expected_splits} < {DEFAULT_EXPANDING_SPLITS})."
    )

oof_probas = np.full(len(X_train_full), np.nan, dtype=float)
oof_fold_ids = np.full(len(X_train_full), -1, dtype=int)

for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_train_full), start=1):
    xgb_model = create_xgb_clone()
    xgb_model.fit(X_train_full.iloc[tr_idx], y_train_binary[tr_idx])
    fold_proba = xgb_model.predict_proba(X_train_full.iloc[va_idx])[:, 1]
    oof_probas[va_idx] = fold_proba
    oof_fold_ids[va_idx] = fold_idx

# Fill any remaining gaps (e.g., tail samples) with predictions from the refit estimator
missing_mask = np.isnan(oof_probas)
if missing_mask.any():
    backup_proba = xgb_best_estimator.predict_proba(X_train_full.iloc[missing_mask])[:, 1]
    oof_probas[missing_mask] = backup_proba
    oof_fold_ids[missing_mask] = 0  # mark as filled from refit model

train_signal_features = pd.DataFrame({
    'xgb_proba': oof_probas,
    'xgb_confidence': np.abs(oof_probas - 0.5) * 2,
    'day_ahead_price': train_df[DAY_AHEAD_COL].values,
    'spot_price': train_df[SPOT_COL].values,
}, index=train_df.index)
train_signal_target = y_train_signed

# Prepare validation features using the refit estimator (trained on full training split)
X_val = val_df[xgb_feature_columns].fillna(0)
y_val_signed = val_df[TARGET_COLUMN].astype(int)
val_probas = xgb_best_estimator.predict_proba(X_val)[:, 1]
val_signal_features = pd.DataFrame({
    'xgb_proba': val_probas,
    'xgb_confidence': np.abs(val_probas - 0.5) * 2,
    'day_ahead_price': val_df[DAY_AHEAD_COL].values,
    'spot_price': val_df[SPOT_COL].values,
}, index=val_df.index)
val_signal_target = y_val_signed

# Fit a fresh XGBoost model on train+validation to generate test-set forecasts
X_full = pd.concat([X_train_full, X_val], axis=0)
y_full_binary = map_target_to_binary(pd.concat([train_signal_target, val_signal_target], axis=0))

xgb_full_model = create_xgb_clone()
xgb_full_model.fit(X_full, y_full_binary)

X_test = test_df[xgb_feature_columns].fillna(0)
y_test_signed = test_df[TARGET_COLUMN].astype(int)
test_probas = xgb_full_model.predict_proba(X_test)[:, 1]

test_signal_features = pd.DataFrame({
    'xgb_proba': test_probas,
    'xgb_confidence': np.abs(test_probas - 0.5) * 2,
    'day_ahead_price': test_df[DAY_AHEAD_COL].values,
    'spot_price': test_df[SPOT_COL].values,
}, index=test_df.index)
test_signal_target = y_test_signed

print(f"Prepared signal datasets for '{signal_dataset_name}' with features: {list(train_signal_features.columns)}")
print(f"Train samples: {len(train_signal_features)}, Validation samples: {len(val_signal_features)}, Test samples: {len(test_signal_features)}")


### Train LightGBM Signal Layer

We detect the appropriate LightGBM execution device and launch a profiled grid search that turns XGBoost probabilities into multi-class trading signals.

**Process:**
```python
# Generate XGBoost probabilities
train_probs = champion_model.predict_proba(X_train)
val_probs = champion_model.predict_proba(X_val)
test_probs = champion_model.predict_proba(X_test)

# Combine with price features
X_train_stacked = np.hstack([train_probs, X_train[price_cols]])
X_val_stacked = np.hstack([val_probs, X_val[price_cols]])
X_test_stacked = np.hstack([test_probs, X_test[price_cols]])

# Initialize LightGBM
lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'device': 'gpu' if torch.cuda.is_available() else 'cpu',
    'seed': SEED
}

# Grid search over LightGBM params
param_grid = {
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_samples': [10, 20, 50]
}

lgb_search = GridSearchCV(
    lgb.LGBMClassifier(**lgb_params),
    param_grid,
    cv=ExpandingWindowSplitter(n_splits=3),
    scoring='accuracy'
)

with profile_stage('LightGBM_tuning'):
    lgb_search.fit(X_train_stacked, y_train)

# Best model
signal_model = lgb_search.best_estimator_
val_score_stacked = signal_model.score(X_val_stacked, y_val)
```

**Device Configuration:**
- **GPU:** LightGBM's `device='gpu'` requires OpenCL; if unavailable, falls back to CPU
- **CPU:** Multi-threaded by default (`n_jobs=-1`)

**Typical Performance Lift:**
- XGBoost alone: 65% accuracy
- LightGBM stacked: 68-72% accuracy (3-7% absolute improvement)

**Output:**
- `signal_model`: Trained LightGBM classifier
- `val_score_stacked`: Validation accuracy of the stacked model

In [ ]:
lgbm_device_config = detect_compute_device(task='training', verbose=True)
print(
    f"LightGBM device: {lgbm_device_config.get('description')} • backend={lgbm_device_config.get('lgbm_device', lgbm_device_config.get('device'))} • jobs={lgbm_device_config.get('n_jobs')}"
)

with StageProfiler("Stage 3 — LightGBM signal model", device_config):
    # Train LightGBM decision model with GridSearchCV on OOF-augmented features
    label_encoder = LabelEncoder()
    label_encoder.fit([-1, 0, 1])  # ensure consistent mapping across all splits

    y_train_encoded = label_encoder.transform(train_signal_target)

    def build_lgbm_classifier(random_state: int = 42) -> lgb.LGBMClassifier:
        params = {
            'objective': 'multiclass',
            'num_class': len(label_encoder.classes_),
            'boosting_type': 'gbdt',
            'random_state': random_state,
            'verbosity': -1,
            'n_jobs': lgbm_device_config.get('n_jobs', -1)
        }
        lgbm_device = lgbm_device_config.get('lgbm_device') or ('gpu' if lgbm_device_config.get('device') == 'cuda' else 'cpu')
        if lgbm_device == 'gpu':
            params.update({
                'device': 'gpu',
                'gpu_platform_id': lgbm_device_config.get('gpu_platform_id', 0),
                'gpu_device_id': lgbm_device_config.get('gpu_device_id', 0)
            })
        else:
            params['device'] = 'cpu'
        return lgb.LGBMClassifier(**params)

    lgbm_param_grid = {
        'num_leaves': [31, 63],
        'max_depth': [-1, 8],
        'learning_rate': [0.02, 0.05],
        'n_estimators': [200, 400],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }

    available_splits = splitter.get_n_splits(train_signal_features)
    cv_splits = min(5, available_splits)
    if cv_splits < 2:
        raise ValueError("Insufficient data for LightGBM cross-validation. Try reducing min_train_size or step size.")

    lgbm_cv = ExpandingWindowSplitter(
        n_splits=cv_splits,
        step_size=DEFAULT_EXPANDING_STEP,
        min_train_size=DEFAULT_MIN_TRAIN_SIZE
    )

    signal_lgbm_estimator = build_lgbm_classifier(random_state=RANDOM_STATE)

    if lgbm_device_config.get('lgbm_device') == 'gpu' or lgbm_device_config.get('device') == 'cuda':
        grid_n_jobs = 1  # avoid oversubscribing GPU
    else:
        grid_n_jobs = lgbm_device_config.get('n_jobs', -1)

    signal_lgbm_grid = GridSearchCV(
        estimator=signal_lgbm_estimator,
        param_grid=lgbm_param_grid,
        scoring='f1_macro',
        cv=lgbm_cv,
        n_jobs=grid_n_jobs,
        verbose=1,
        refit=True,
        return_train_score=True
    )

    signal_lgbm_grid.fit(train_signal_features, y_train_encoded)

    signal_best_lgbm = signal_lgbm_grid.best_estimator_
    print("Best LightGBM params:", signal_lgbm_grid.best_params_)
    print(f"Best CV macro-F1: {signal_lgbm_grid.best_score_:.4f}")

    signal_model_artifacts = {
        'label_encoder': label_encoder,
        'grid_search': signal_lgbm_grid,
        'best_estimator': signal_best_lgbm,
        'feature_names': list(train_signal_features.columns),
        'device_config': lgbm_device_config
    }


### Benchmark Price-Only Baseline

Training a baseline LightGBM on price features alone quantifies how much lift the news-enhanced signal layer provides.

**Baseline Configuration:**
```python
# Use only price features (no XGBoost probabilities from news-enhanced model)
X_train_baseline = X_train[price_cols]  # e.g., price_lag_1h, price_ma_24h
X_val_baseline = X_val[price_cols]

# Train LightGBM with same hyperparameters as stacked model
baseline_model = lgb.LGBMClassifier(**lgb_search.best_params_, seed=SEED)
baseline_model.fit(X_train_baseline, y_train)

val_score_baseline = baseline_model.score(X_val_baseline, y_val)
```

**Comparison:**
```python
print(f"Baseline (price-only): {val_score_baseline:.4f}")
print(f"Signal (news-enhanced): {val_score_stacked:.4f}")
print(f"Lift: {(val_score_stacked - val_score_baseline) * 100:.2f}%")
```

**Interpretation:**
- **Lift <2%:** News features add little value (maybe not worth the NLP compute cost)
- **Lift 3-7%:** Significant improvement (worth deploying)
- **Lift >10%:** Strong signal (verify no data leakage)

**Why This Matters for Business:**
If the baseline achieves 70% accuracy and the news-enhanced model achieves 72%, the question becomes: "Is 2% better decisions worth running a news scraper and embedding models?" This comparison informs deployment priorities.

**Statistical Testing:**
In the next section, we'll run McNemar's test to confirm the difference is statistically significant (not just luck).

In [ ]:
with StageProfiler("Stage 3 — LightGBM baseline model", device_config):
    # Train baseline LightGBM without news-driven features (price-only inputs)
    baseline_feature_cols = ['day_ahead_price', 'spot_price']

    train_baseline_features = train_signal_features[baseline_feature_cols].copy()
    val_baseline_features = val_signal_features[baseline_feature_cols].copy()
    test_baseline_features = test_signal_features[baseline_feature_cols].copy()

    y_train_baseline = label_encoder.transform(train_signal_target)

    baseline_lgbm_estimator = build_lgbm_classifier(random_state=RANDOM_STATE)

    baseline_param_grid = {
        'num_leaves': [15, 31, 63],
        'max_depth': [-1, 8],
        'learning_rate': [0.02, 0.05, 0.1],
        'n_estimators': [200, 400],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }

    baseline_cv_splits = min(5, available_splits)
    baseline_cv = ExpandingWindowSplitter(
        n_splits=baseline_cv_splits,
        step_size=DEFAULT_EXPANDING_STEP,
        min_train_size=DEFAULT_MIN_TRAIN_SIZE
    )

    baseline_grid = GridSearchCV(
        estimator=baseline_lgbm_estimator,
        param_grid=baseline_param_grid,
        scoring='f1_macro',
        cv=baseline_cv,
        n_jobs=grid_n_jobs,
        verbose=1,
        refit=True,
        return_train_score=True
    )

    baseline_grid.fit(train_baseline_features, y_train_baseline)

    baseline_best_lgbm = baseline_grid.best_estimator_
    print("Baseline LightGBM params:", baseline_grid.best_params_)
    print(f"Baseline CV macro-F1: {baseline_grid.best_score_:.4f}")

    baseline_model_artifacts = {
        'grid_search': baseline_grid,
        'best_estimator': baseline_best_lgbm,
        'feature_names': baseline_feature_cols,
        'device_config': lgbm_device_config
    }


### Flush Telemetry Logs

We persist the telemetry buffer so resource traces and metadata survive kernel restarts.

**What Gets Flushed:**
1. **Profiling Data:**
   - Stage names, durations, peak memory, disk I/O
   - Aggregated stats (total runtime, bottleneck stages)

2. **Model Metadata:**
   - Champion XGBoost params
   - LightGBM params
   - Feature lists

3. **Performance Metrics:**
   - CV scores, validation scores, test scores (when available)
   - Per-class precision/recall

4. **System Info:**
   - GPU model and memory
   - CPU count and RAM
   - Python/package versions

**Output Format:**
JSON file for programmatic analysis:
```json
{
  "session_id": "2024-01-15_14-30-22",
  "total_runtime_minutes": 127.3,
  "stages": [
    {"name": "feature_engineering", "duration_sec": 342, "peak_ram_gb": 4.2},
    {"name": "xgboost_tuning", "duration_sec": 7234, "peak_gpu_mem_gb": 6.1},
    ...
  ],
  "champion_model": {
    "param_id": "lambda_0.95_horizon_48",
    "val_accuracy": 0.6587,
    "xgboost_params": {...}
  },
  "signal_model": {
    "val_accuracy": 0.6891,
    "lgbm_params": {...}
  }
}
```

**Why Persist Now:**
If the kernel crashes during final evaluation, we don't lose all metadata. The telemetry file becomes the source of truth for experiment tracking.

**Access Later:**
```python
import json
with open('logs/telemetry_2024-01-15_14-30-22.json') as f:
    telemetry = json.load(f)
print(f"Best val score: {telemetry['signal_model']['val_accuracy']}")
```

In [ ]:
if 'telemetry_logger' in globals():
    log_path = telemetry_logger.flush()
    print(f"Telemetry written to {log_path}")


## 📈 Confusion Matrix Visualizations

Visual confusion matrices help us confirm whether misclassifications cluster in specific regimes, informing whether we need additional features or decision thresholds.

**What is a Confusion Matrix (Revisited)?**
For each pair of (actual, predicted) classes, count occurrences:
```
                Predicted
              Long Neutral Short
Actual Long    142    23      15     ← 142 correct, 38 errors
       Neutral  19   238      31
       Short    12    28     145
```

**Key Diagnostics:**
1. **Diagonal Dominance:** High values on diagonal = good accuracy
2. **Systematic Bias:** If Neutral column has high values → model over-predicts Neutral
3. **Asymmetric Errors:**
   - Long misclassified as Short (or vice versa): Very bad for trading (reverses position)
   - Long misclassified as Neutral: Missed profit (less costly)

**Visualization Enhancements:**
- **Heatmap:** Color intensity shows count magnitude
- **Annotations:** Display counts and percentages in each cell
- **Normalization:** Show row-wise percentages (e.g., "Of 100 actual Longs, 79% predicted correctly")

**Comparisons:**
Plot confusion matrices for:
1. **Baseline (price-only):** Understand where price signals fail
2. **Signal (news-enhanced):** Check if news reduces specific error types
3. **Naive (always predict most common class):** Sanity check

**Insight Example:**
If baseline confuses Long/Short (off-diagonal corners) but signal model reduces these errors, news features help with directional classification. If errors cluster in Neutral→Long/Short, maybe the neutral band threshold needs adjustment.

In [ ]:
print("Generating Confusion Matrices...\n")

models_to_evaluate = {}

if 'signal_best_lgbm' in locals() and signal_best_lgbm is not None:
    models_to_evaluate['LightGBM Signal'] = (
        signal_best_lgbm,
        test_signal_features  # features created in the signal-model prep cell
    )

if 'baseline_best_lgbm' in locals() and baseline_best_lgbm is not None:
    models_to_evaluate['LightGBM Baseline'] = (
        baseline_best_lgbm,
        test_baseline_features  # ['day_ahead_price', 'spot_price']
    )

if 'best_xgb_model_selection' in locals() and best_xgb_model_selection is not None:
    print("Skipping XGBoost confusion matrix: binary forecasts are not comparable to 3-class targets.")

if models_to_evaluate:
    X_test_eval = test_df.drop(columns=[TARGET_COLUMN])
    y_test_eval = test_df[TARGET_COLUMN]
    plot_confusion_matrices(
        models_to_evaluate,
        y_test_eval,
        class_labels=[-1, 0, 1],
        label_encoder=label_encoder
    )
else:
    print("⚠ No models available for confusion matrix analysis")

## 🔬 Statistical Significance Testing

We complement raw accuracy with statistical tests to ensure the observed improvements are not due to sampling noise.

**Why Accuracy Alone Isn't Enough:**
- Baseline: 68.2% accuracy (341/500 test samples correct)
- Signal: 70.4% accuracy (352/500 test samples correct)
- Difference: 2.2% (11 more correct predictions)

Is 11 samples significant, or could random noise produce this difference?

**Bootstrap Confidence Intervals:**
Resample test set 1000 times with replacement, recompute accuracy each time:
```python
bootstrap_scores = []
for _ in range(1000):
    sample_idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    score = (y_pred[sample_idx] == y_test[sample_idx]).mean()
    bootstrap_scores.append(score)

ci_lower, ci_upper = np.percentile(bootstrap_scores, [2.5, 97.5])
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
```

**Interpretation:**
- Baseline CI: [0.652, 0.712]
- Signal CI: [0.674, 0.734]
- Overlap: [0.674, 0.712] → difference may be marginal

**McNemar's Test:**
Specifically for paired classifiers (same test set):
```python
from statsmodels.stats.contingency_tables import mcnemar

# Contingency table:
# | Baseline Correct | Baseline Wrong |
# |-----------------|----------------|
# | Signal Correct  |      a         |      b         |
# | Signal Wrong    |      c         |      d         |

# McNemar focuses on b and c (disagreements)
table = [[a, b], [c, d]]
result = mcnemar(table, correction=True)
print(f"p-value: {result.pvalue:.4f}")
```

**Decision Rule:**
- p < 0.05: Signal model is significantly better (reject null hypothesis that models perform equally)
- p ≥ 0.05: Difference could be noise (don't deploy based on this alone)

**Trading Context:**
Even a 2% accuracy improvement can be valuable if it translates to profitable trades (backtest on actual returns, not just classification accuracy).

In [ ]:
print("Performing Statistical Significance Testing...\n")

if 'signal_best_lgbm' in locals() and 'baseline_best_lgbm' in locals():
    if signal_best_lgbm is not None and baseline_best_lgbm is not None:
        y_test_eval = test_signal_target  # or test_df[TARGET_COLUMN] if already aligned

        # use the feature sets each model trained on
        X_test_signal = test_signal_features
        X_test_baseline = test_baseline_features

        signal_pred_encoded = signal_best_lgbm.predict(X_test_signal)
        baseline_pred_encoded = baseline_best_lgbm.predict(X_test_baseline)

        try:
            signal_proba = signal_best_lgbm.predict_proba(X_test_signal)
        except LightGBMError:
            signal_proba = None

        try:
            baseline_proba = baseline_best_lgbm.predict_proba(X_test_baseline)
        except LightGBMError:
            baseline_proba = None

        signal_predictions = label_encoder.inverse_transform(signal_pred_encoded)
        baseline_predictions = label_encoder.inverse_transform(baseline_pred_encoded)

        statistical_results = compare_models_statistically(
            y_test_eval,
            signal_predictions,
            baseline_predictions,
            signal_proba=signal_proba,
            baseline_proba=baseline_proba
        )

        print("\nStatistical test results saved to 'statistical_results' variable")
    else:
        print("⚠ One or both LightGBM models are None")
else:
    print("⚠ Both signal and baseline models needed for statistical comparison")

### Evaluate Signal vs Baselines

We compare classification metrics, ROC curves, and trading returns for signal, baseline, and naive strategies to understand the real business impact.

**Classification Metrics:**
```python
from sklearn.metrics import classification_report, roc_auc_score

for name, model, X in [('Baseline', baseline_model, X_test_baseline),
                        ('Signal', signal_model, X_test_stacked)]:
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    print(f"
{name} Model:")
    print(classification_report(y_test, y_pred, target_names=['Long', 'Neutral', 'Short']))
    print(f"Macro-avg ROC-AUC: {roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro'):.4f}")
```

**Example Output:**
```
Baseline Model:
              precision    recall  f1-score   support
Long              0.68      0.71      0.69       180
Neutral           0.72      0.69      0.70       288
Short             0.65      0.67      0.66       185

Macro-avg ROC-AUC: 0.7823

Signal Model:
              precision    recall  f1-score   support
Long              0.71      0.74      0.72       180
Neutral           0.74      0.71      0.72       288
Short             0.68      0.71      0.69       185

Macro-avg ROC-AUC: 0.8041
```

**Trading Returns Simulation:**
```python
def simulate_trading(y_true, y_pred, spread_actual):
    """
    Simulate profit/loss based on predictions.
    - Long: Earn +spread if spread widens (y_true=Long)
    - Short: Earn +spread if spread narrows (y_true=Short)
    - Neutral: No position (0 return)
    """
    returns = []
    for true, pred, spread in zip(y_true, y_pred, spread_actual):
        if pred == 'Long' and true == 'Long':
            returns.append(spread)
        elif pred == 'Short' and true == 'Short':
            returns.append(spread)
        elif pred == 'Neutral':
            returns.append(0)
        else:
            returns.append(-spread)  # Wrong direction
    return np.sum(returns), np.mean(returns)

baseline_pnl, baseline_avg = simulate_trading(y_test, y_pred_baseline, spreads_test)
signal_pnl, signal_avg = simulate_trading(y_test, y_pred_signal, spreads_test)

print(f"Baseline Total P&L: {baseline_pnl:.2f} EUR")
print(f"Signal Total P&L: {signal_pnl:.2f} EUR")
print(f"Improvement: {signal_pnl - baseline_pnl:.2f} EUR ({(signal_pnl/baseline_pnl - 1)*100:.1f}%)")
```

**Business Decision:**
If signal model improves P&L by 15-30%, the operational cost of running news scrapers and NLP models is justified.

In [ ]:
# Evaluate tuned decision model vs baseline and naive strategy on the test set
y_test_encoded = label_encoder.transform(test_signal_target)

signal_test_pred = signal_best_lgbm.predict(test_signal_features)
signal_test_proba = signal_best_lgbm.predict_proba(test_signal_features)

baseline_test_pred = baseline_best_lgbm.predict(test_baseline_features)
baseline_test_proba = baseline_best_lgbm.predict_proba(test_baseline_features)


def _safe_multiclass_auc(y_true: np.ndarray, proba: np.ndarray) -> float:
    unique_classes = np.unique(y_true)
    if unique_classes.size <= 1:
        return np.nan
    if unique_classes.size == 2:
        # Reduce to binary ROC AUC using the higher label as the positive class
        positive_class = unique_classes.max()
        proba_binary = proba[:, positive_class]
        y_binary = (y_true == positive_class).astype(int)
        return roc_auc_score(y_binary, proba_binary)
    proba_aligned = proba[:, unique_classes]
    return roc_auc_score(
        y_true,
        proba_aligned,
        multi_class='ovo',
        average='macro',
        labels=unique_classes
    )

# Classification metrics
signal_metrics = {
    'Model': 'LightGBM Signal (with news)',
    'Accuracy': accuracy_score(y_test_encoded, signal_test_pred),
    'Macro-F1': f1_score(y_test_encoded, signal_test_pred, average='macro', zero_division=0),
    'AUC (macro)': _safe_multiclass_auc(y_test_encoded, signal_test_proba)
}

baseline_metrics = {
    'Model': 'LightGBM Baseline (price-only)',
    'Accuracy': accuracy_score(y_test_encoded, baseline_test_pred),
    'Macro-F1': f1_score(y_test_encoded, baseline_test_pred, average='macro', zero_division=0),
    'AUC (macro)': _safe_multiclass_auc(y_test_encoded, baseline_test_proba)
}

metrics_df = pd.DataFrame([signal_metrics, baseline_metrics]).set_index('Model')
display(metrics_df)

auc_series = metrics_df['AUC (macro)'].dropna()
observed_labels = np.unique(y_test_encoded)
if observed_labels.size >= 2 and not auc_series.empty:
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import roc_curve

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1, label='Chance')

    model_curves = [
        (signal_test_proba, 'LightGBM Signal (with news)', '#4C72B0'),
        (baseline_test_proba, 'LightGBM Baseline (price-only)', '#55A868')
    ]

    if observed_labels.size == 2:
        positive_class = observed_labels.max()
        y_binary = (y_test_encoded == positive_class).astype(int)
        for proba, label, color in model_curves:
            fpr, tpr, _ = roc_curve(y_binary, proba[:, positive_class])
            ax.plot(fpr, tpr, label=f'{label}', color=color, linewidth=2)
    else:
        y_binarised = label_binarize(y_test_encoded, classes=observed_labels)
        for proba, label, color in model_curves:
            for idx, cls in enumerate(observed_labels):
                fpr, tpr, _ = roc_curve(y_binarised[:, idx], proba[:, cls])
                ax.plot(fpr, tpr, label=f'{label} vs class {label_encoder.inverse_transform([cls])[0]}', color=color, linewidth=1.5, alpha=0.6)

    ax.set_title('ROC Curves (Macro AUC shown in table)')
    ax.set_xlabel('False Positive Rate (FPR)')
    ax.set_ylabel('True Positive Rate (TPR)')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='lower right', fontsize='small')
    fig.tight_layout()
    plt.show()

print("\nClassification report (signal model):")
print(
    classification_report(
        y_test_encoded,
        signal_test_pred,
        labels=observed_labels,
        target_names=[str(cls) for cls in label_encoder.inverse_transform(observed_labels)]
    )
)

# Trading performance metrics
spot_series = test_df[SPOT_COL]
day_ahead_series = test_df[DAY_AHEAD_COL]
spread_series = spot_series - day_ahead_series

signal_actions = label_encoder.inverse_transform(signal_test_pred)
baseline_actions = label_encoder.inverse_transform(baseline_test_pred)
naive_actions = np.ones_like(signal_actions, dtype=int)


def actions_to_returns(actions: np.ndarray, spread: pd.Series) -> pd.Series:
    returns = np.where(actions == 1, spread, np.where(actions == -1, -spread, 0.0))
    return pd.Series(returns, index=spread.index)

signal_returns = actions_to_returns(signal_actions, spread_series)
baseline_returns = actions_to_returns(baseline_actions, spread_series)
naive_returns = actions_to_returns(naive_actions, spread_series)


def summarise_returns(returns: pd.Series, label: str) -> dict:
    cumulative = returns.cumsum()
    drawdown = cumulative - cumulative.cummax()
    mean_return = returns.mean()
    std_return = returns.std(ddof=1)
    periods_per_year = 24 * 365  # assume hourly cadence
    sharpe = (mean_return / std_return * np.sqrt(periods_per_year)) if std_return > 0 else np.nan
    return {
        'Strategy': label,
        'Total Return': cumulative.iloc[-1],
        'Average Return': mean_return,
        'Volatility': std_return,
        'Sharpe (annualised)': sharpe,
        'Max Drawdown': drawdown.min()
    }

returns_summary = pd.DataFrame([
    summarise_returns(signal_returns, 'LightGBM Signal'),
    summarise_returns(baseline_returns, 'LightGBM Baseline'),
    summarise_returns(naive_returns, 'Naive Buy-DA/Sell-Spot')
]).set_index('Strategy')

display(returns_summary)

# Plot cumulative returns comparison

def _prepare_plot_series(returns: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    cumulative = returns.cumsum()
    index = cumulative.index
    if isinstance(index, pd.DatetimeIndex):
        if index.tz is not None:
            index = index.tz_convert(None)
        x_values = index.to_pydatetime()
    else:
        x_values = np.arange(len(cumulative))
    return x_values, cumulative.to_numpy()

signal_x, signal_y = _prepare_plot_series(signal_returns)
baseline_x, baseline_y = _prepare_plot_series(baseline_returns)
naive_x, naive_y = _prepare_plot_series(naive_returns)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(signal_x, signal_y, '-', label='LightGBM Signal (with news)', linewidth=2)
ax.plot(baseline_x, baseline_y, '--', label='LightGBM Baseline (price-only)')
ax.plot(naive_x, naive_y, ':', label='Naive Buy-DA/Sell-Spot')
ax.set_title('Cumulative Strategy Returns on Test Set')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Cumulative Return (currency units)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# Store artifacts for downstream analysis
evaluation_artifacts = {
    'classification_metrics': metrics_df,
    'returns_summary': returns_summary,
    'signal_returns': signal_returns,
    'baseline_returns': baseline_returns,
    'naive_returns': naive_returns,
    'signal_actions': signal_actions,
    'baseline_actions': baseline_actions,
    'naive_actions': naive_actions
}


### Interpret LightGBM Signals

We inspect feature importances to identify which inputs drive the signal and baseline decisions, guiding future feature engineering.

**LightGBM Feature Importance:**
Tree-based models track how often each feature is used for splits and how much it improves predictions:
- **Gain:** Total reduction in loss from splits using this feature
- **Split:** Number of times feature was used to split nodes
- **Cover:** Average number of samples affected by splits (less common metric)

**Extracting Importance:**
```python
import matplotlib.pyplot as plt

# Get feature importances
importances = signal_model.feature_importances_
feature_names = ['P(Long)', 'P(Neutral)', 'P(Short)', 'price_current', 'price_lag_1h', ...]

# Sort and plot
sorted_idx = np.argsort(importances)[::-1][:10]  # Top 10
plt.barh(range(10), importances[sorted_idx])
plt.yticks(range(10), [feature_names[i] for i in sorted_idx])
plt.xlabel('Feature Importance (Gain)')
plt.title('LightGBM Signal Model - Top 10 Features')
plt.show()
```

**Expected Patterns:**
1. **XGBoost Probabilities Dominate:** `P(Long)`, `P(Short)` should be top features (validates stacking approach)
2. **Price Context Matters:** `price_current` or `price_ma_24h` refines decisions (confirms risk-adjustment hypothesis)
3. **Confidence Metrics:** If `max(P)` or `entropy(P)` ranks high, model uses XGBoost confidence to gate trades

**Insights for Iteration:**
- **If `P(Long)` is only important feature:** Price context adds no value → simplify to XGBoost-only pipeline
- **If `price_lag_1h` dominates:** Recent price is more predictive than spread forecasts → revisit XGBoost features
- **If all importances are uniform:** LightGBM struggles to find signal → try more expressive features or nonlinear stacking (neural net)

**Baseline Comparison:**
Compare baseline's feature importance (price features only) to signal's importance to see if news-derived probabilities actually contribute.

In [ ]:
print("Analyzing LightGBM feature importance…\n")

# Signal model (trained on train_signal_features)
if 'signal_best_lgbm' in locals() and signal_best_lgbm is not None:
    print("=== Signal Model ===")
    signal_feature_cols = list(train_signal_features.columns)
    signal_importance_df = plot_feature_importance(
        signal_best_lgbm,
        signal_feature_cols,
        model_name='LightGBM Signal Model',
        top_n=20
    )
else:
    print("Signal model artifacts not available.\n")

# Baseline model (trained on baseline feature subset)
if 'baseline_best_lgbm' in locals() and baseline_best_lgbm is not None:
    print("\n=== Baseline Model ===")
    baseline_feature_cols = list(test_baseline_features.columns)
    baseline_importance_df = plot_feature_importance(
        baseline_best_lgbm,
        baseline_feature_cols,
        model_name='LightGBM Baseline Model',
        top_n=20
    )
else:
    print("\nBaseline model artifacts not available.")

### Inspect XGBoost Drivers

Evaluating XGBoost feature importance reveals which engineered time-decay features influence the upstream forecaster, informing potential refinements.

**XGBoost Feature Importance:**
Similar to LightGBM, but now we're examining the 60-100 input features:
```python
xgb_importances = champion_model.feature_importances_
feature_names_xgb = list(X_train.columns)  # All baseline + news features

sorted_idx = np.argsort(xgb_importances)[::-1][:20]  # Top 20
plt.figure(figsize=(10, 8))
plt.barh(range(20), xgb_importances[sorted_idx])
plt.yticks(range(20), [feature_names_xgb[i] for i in sorted_idx])
plt.xlabel('Feature Importance (Gain)')
plt.title('XGBoost Spread Forecaster - Top 20 Features')
plt.show()
```

**Key Questions:**
1. **Do News Features Appear?**
   - If `supply_shock_decayed` or `embedding_umap_3` rank in top 10 → news adds value
   - If only price lags dominate → news is redundant

2. **Which Decay Settings Work?**
   - If `_lambda_0.95_` features rank higher than `_lambda_0.90_` → medium decay optimal
   - If `_horizon_72h_` beats `_horizon_24h_` → longer history matters

3. **Are Embeddings Used?**
   - If UMAP dimensions appear → semantic content matters
   - If only topic counts appear → discrete labels suffice (can skip embedding generation for speed)

**Refinement Ideas:**
- **Low-Importance Features:** Drop features with <1% importance to speed up training
- **High-Importance Clusters:** Engineer interaction features (e.g., `supply_shock × price_volatility`)
- **Redundancy:** If `price_lag_1h` and `price_lag_2h` both appear, maybe just keep one

**Cross-Validation:**
Check if importance rankings are stable across CV folds. If `supply_shock` is top feature in one fold but absent in another, it may be overfitting to specific market regimes.

In [ ]:
# Analyze XGBoost feature importance
if 'best_xgb_model_selection' in locals() and best_xgb_model_selection is not None:
    print("\nAnalyzing XGBoost Model Feature Importance...")
    feature_cols = [col for col in train_df.columns if col != TARGET_COLUMN]
    xgb_importance = plot_feature_importance(
        best_xgb_model_selection,
        feature_cols,
        'XGBoost Model',
        top_n=20
    )
